# Offline Multi-Class Flight Bookability Prediction (Colab Review Notebook)

**Project Goal:** Build and evaluate an offline or batch multiclass flight bookability prediction pipeline for metasearch offers.

This notebook is organized as an examiner-friendly code walkthrough for Google Colab. The code is broken into granular sections so the implementation can be read, downloaded, and reviewed comfortably without needing to jump around the repository.

### What your mentor can review here
- schema validation and label engineering
- corrected preprocessing logic
- valid feature engineering only
- leakage-safe temporal splitting
- model comparison and calibrator comparison
- evaluation metrics and reliability diagnostics
- ablation support and operational policy simulation
- final training and inference entry points


## Colab Setup

If this notebook is opened in Google Colab, the following optional setup cell can be used to install the main dependencies. It is safe to skip if the notebook is being reviewed rather than executed.


In [ ]:
# Optional Colab setup
# Uncomment this cell if you want to run the code in Google Colab.

# !pip install pandas numpy scikit-learn catboost matplotlib seaborn streamlit nbformat


## Review Notes

- The notebook mirrors the current repository state as of generation time.
- `src/` contains the main implementation.
- `src/` now contains the final coherent pipeline.
- Older prototype code is intentionally not included in this notebook because it is no longer the authoritative path.
- Some scripts expect local project files such as `data/raw/...`, `data/processed/...`, and `artifacts/...`.
- The notebook is structured for review first and execution second.


## Configuration

**Source file:** `src/config.py`

This module defines the central project configuration, path management, taxonomy settings, and the documented prediction-time assumption.

### Imports, constants, and top-level configuration

This block contains the imports, module-level constants, and shared state used by `src/config.py`.

In [ ]:
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List


### Conditional Block: `'__file__' in globals()`

This block is extracted directly from `src/config.py` so the implementation can be reviewed in smaller steps.

In [ ]:
if "__file__" in globals():
    ROOT_DIR = Path(__file__).resolve().parents[1]
else:
    ROOT_DIR = Path.cwd()


### Class: `ProjectConfig`

This block is extracted directly from `src/config.py` so the implementation can be reviewed in smaller steps.

In [ ]:
class ProjectConfig:
    """Central configuration used by the final research pipeline.

    The goal of this config object is to keep the pipeline transparent and easy
    to audit. The prediction timestamp assumption is especially important:
    unless a stronger real-time event timestamp exists in the raw logs, we treat
    `LandingTime` as the decision-time proxy.
    """

    root_dir: Path = ROOT_DIR
    raw_data_path: Path = ROOT_DIR / "data" / "raw" / "tbl_SearchTracking_Merged.csv"
    processed_data_path: Path = ROOT_DIR / "data" / "processed" / "final_processed_flight_data.csv"
    artifacts_dir: Path = ROOT_DIR / "artifacts"
    models_dir: Path = ROOT_DIR / "artifacts" / "models"
    reports_dir: Path = ROOT_DIR / "artifacts" / "reports"
    notebook_path: Path = ROOT_DIR / "notebooks" / "Flight_Bookability_Analysis_V2.ipynb"

    prediction_time_field_name: str = "prediction_time"
    prediction_time_assumption: str = (
        "LandingTime is used as the prediction-time proxy because it is the "
        "best available timestamp for the real-time decision moment in the "
        "current raw log export."
    )

    train_fraction: float = 0.60
    validation_fraction: float = 0.20
    test_fraction: float = 0.20
    rolling_backtest_windows: int = 3

    random_state: int = 42
    calibration_alpha: float = 0.05

    final_target_labels: List[str] = field(
        default_factory=lambda: [
            "bookable",
            "price_changed",
            "unavailable",
            "technical_failure",
        ]
    )

    column_aliases: Dict[str, List[str]] = field(
        default_factory=lambda: {
            "status": ["Status", "status", "booking_status", "result_status"],
            "origin": ["Origin", "origin", "from", "origin_airport"],
            "destination": ["Destination", "destination", "to", "destination_airport"],
            "airline": ["Airline", "airline", "carrier", "airline_code"],
            "cabin": ["Class", "class", "cabin", "cabin_class"],
            "departure_date": ["DepDate", "departure_date", "dep_date", "flight_date"],
            "prediction_time": ["LandingTime", "prediction_time", "search_time", "event_time"],
            "inserted_on": ["InsertedOn", "inserted_on", "outcome_time"],
            "trip_type": ["SearchType", "trip_type", "search_type"],
            "previous_page": ["PreviousPage", "previous_page", "referrer", "url"],
            "landing_page": ["LandingPage", "landing_page"],
            "fdtag": ["FdTag", "fdtag"],
            "user_agent": ["UserAgent", "user_agent"],
            "flight_segments": ["FlightSegments", "flight_segments"],
            "provider_id": ["Provider", "provider", "provider_id"],
            "price": ["Price", "price", "total_price", "fare"],
            "currency": ["Currency", "currency"],
            "stops": ["Stops", "stops", "stop_count"],
            "market": ["Market", "market"],
            "locale": ["Locale", "locale"],
            "device": ["Device", "device"],
            "adults": ["Adults", "adults", "adult_count"],
            "children": ["Children", "children", "child_count"],
            "infants": ["Infants", "infants", "infant_count"],
        }
    )

    required_logical_fields: List[str] = field(
        default_factory=lambda: [
            "status",
            "origin",
            "destination",
            "airline",
            "cabin",
            "departure_date",
            "prediction_time",
        ]
    )

    optional_logical_fields: List[str] = field(
        default_factory=lambda: [
            "inserted_on",
            "trip_type",
            "previous_page",
            "landing_page",
            "fdtag",
            "user_agent",
            "flight_segments",
            "provider_id",
            "price",
            "currency",
            "stops",
            "market",
            "locale",
            "device",
            "adults",
            "children",
            "infants",
        ]
    )

    def ensure_directories(self) -> None:
        self.artifacts_dir.mkdir(parents=True, exist_ok=True)
        self.models_dir.mkdir(parents=True, exist_ok=True)
        self.reports_dir.mkdir(parents=True, exist_ok=True)
        self.processed_data_path.parent.mkdir(parents=True, exist_ok=True)
        self.notebook_path.parent.mkdir(parents=True, exist_ok=True)

    def to_dict(self) -> Dict[str, object]:
        return {
            "root_dir": str(self.root_dir),
            "raw_data_path": str(self.raw_data_path),
            "processed_data_path": str(self.processed_data_path),
            "artifacts_dir": str(self.artifacts_dir),
            "models_dir": str(self.models_dir),
            "reports_dir": str(self.reports_dir),
            "notebook_path": str(self.notebook_path),
            "prediction_time_field_name": self.prediction_time_field_name,
            "prediction_time_assumption": self.prediction_time_assumption,
            "train_fraction": self.train_fraction,
            "validation_fraction": self.validation_fraction,
            "test_fraction": self.test_fraction,
            "rolling_backtest_windows": self.rolling_backtest_windows,
            "random_state": self.random_state,
            "calibration_alpha": self.calibration_alpha,
            "final_target_labels": list(self.final_target_labels),
            "column_aliases": dict(self.column_aliases),
            "required_logical_fields": list(self.required_logical_fields),
            "optional_logical_fields": list(self.optional_logical_fields),
        }


## Utility Helpers

**Source file:** `src/utils.py`

Shared helper functions used across schema handling, URL parsing, numeric conversion, text normalization, and artifact writing.

### Imports, constants, and top-level configuration

This block contains the imports, module-level constants, and shared state used by `src/utils.py`.

In [ ]:
import json
import re
from pathlib import Path
from typing import Any, Dict, Iterable, Optional
from urllib.parse import parse_qs, urlparse

import numpy as np
import pandas as pd


### Function: `normalize_name`

This block is extracted directly from `src/utils.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def normalize_name(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(name).strip().lower())


### Function: `safe_to_datetime`

This block is extracted directly from `src/utils.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def safe_to_datetime(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, errors="coerce", format="mixed")


### Function: `safe_to_numeric`

This block is extracted directly from `src/utils.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def safe_to_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


### Function: `normalize_text`

This block is extracted directly from `src/utils.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def normalize_text(series: pd.Series, default: str = "Unknown") -> pd.Series:
    cleaned = series.astype("string").fillna(default).str.strip()
    cleaned = cleaned.replace({"": default, "<NA>": default, "nan": default, "None": default})
    return cleaned.fillna(default)


### Function: `extract_domain`

This block is extracted directly from `src/utils.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def extract_domain(value: Any, default: str = "Unknown") -> str:
    if pd.isna(value):
        return default
    text = str(value).strip()
    if not text:
        return default
    try:
        parsed = urlparse(text)
        domain = parsed.netloc.lower().replace("www.", "")
        return domain or default
    except Exception:
        return default


### Function: `parse_query_params`

This block is extracted directly from `src/utils.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def parse_query_params(url: Any) -> Dict[str, str]:
    if pd.isna(url):
        return {}
    try:
        parsed = urlparse(str(url))
        parsed_qs = parse_qs(parsed.query)
        return {k.lower(): v[0] for k, v in parsed_qs.items() if v}
    except Exception:
        return {}


### Function: `first_present`

This block is extracted directly from `src/utils.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def first_present(params: Dict[str, str], keys: Iterable[str], default: Optional[str] = None) -> Optional[str]:
    for key in keys:
        if key in params and params[key] not in (None, ""):
            return params[key]
    return default


### Function: `save_json`

This block is extracted directly from `src/utils.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def save_json(path: Path, payload: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, default=_json_default)


### Function: `save_csv`

This block is extracted directly from `src/utils.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def save_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)


### Function: `_json_default`

This block is extracted directly from `src/utils.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def _json_default(value: Any) -> Any:
    if isinstance(value, (pd.Timestamp, np.datetime64)):
        return str(value)
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    return value


## Schema Resolution

**Source file:** `src/schema.py`

Alias-based schema validation ensures the pipeline does not touch raw columns before confirming the logical fields required by the project.

### Imports, constants, and top-level configuration

This block contains the imports, module-level constants, and shared state used by `src/schema.py`.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional

from src.utils import normalize_name


### Class: `SchemaResolution`

This block is extracted directly from `src/schema.py` so the implementation can be reviewed in smaller steps.

In [ ]:
class SchemaResolution:
    required: Dict[str, str]
    optional: Dict[str, Optional[str]]
    missing_required: List[str]
    matched_aliases: Dict[str, Optional[str]]

    @property
    def is_valid(self) -> bool:
        return not self.missing_required

    def all_columns(self) -> Dict[str, Optional[str]]:
        combined = dict(self.optional)
        combined.update(self.required)
        return combined


### Function: `resolve_schema`

This block is extracted directly from `src/schema.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def resolve_schema(columns: List[str], aliases: Dict[str, List[str]], required_fields: List[str], optional_fields: List[str]) -> SchemaResolution:
    normalized_columns = {normalize_name(column): column for column in columns}

    required: Dict[str, str] = {}
    optional: Dict[str, Optional[str]] = {}
    missing_required: List[str] = []
    matched_aliases: Dict[str, Optional[str]] = {}

    for logical_name in required_fields:
        match, matched_alias = _resolve_single(logical_name, normalized_columns, aliases)
        if match is None:
            missing_required.append(logical_name)
        else:
            required[logical_name] = match
        matched_aliases[logical_name] = matched_alias

    for logical_name in optional_fields:
        match, matched_alias = _resolve_single(logical_name, normalized_columns, aliases)
        optional[logical_name] = match
        matched_aliases[logical_name] = matched_alias

    return SchemaResolution(required=required, optional=optional, missing_required=missing_required, matched_aliases=matched_aliases)


### Function: `_resolve_single`

This block is extracted directly from `src/schema.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def _resolve_single(logical_name: str, normalized_columns: Dict[str, str], aliases: Dict[str, List[str]]) -> tuple[Optional[str], Optional[str]]:
    candidates = aliases.get(logical_name, []) + [logical_name]
    for candidate in candidates:
        normalized = normalize_name(candidate)
        if normalized in normalized_columns:
            return normalized_columns[normalized], candidate
    return None, None


## Canonical Label Engineering

**Source file:** `src/labels.py`

This module maps noisy operational statuses into the final coherent target taxonomy used throughout the final project.

### Imports, constants, and top-level configuration

This block contains the imports, module-level constants, and shared state used by `src/labels.py`.

In [ ]:
from typing import Tuple

import pandas as pd


FINAL_LABELS = ["bookable", "price_changed", "unavailable", "technical_failure"]


### Function: `canonicalize_status`

This block is extracted directly from `src/labels.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def canonicalize_status(value) -> str:
    if pd.isna(value):
        return "ambiguous"

    status = str(value).strip().lower()
    if not status:
        return "ambiguous"

    if any(token in status for token in ["booked", "success", "bookable"]):
        return "bookable"
    if any(token in status for token in ["price mismatch", "price changed", "fare changed", "price mismatch/fare changed"]):
        return "price_changed"
    if any(token in status for token in ["not available", "unavailable", "sold out"]):
        return "unavailable"
    if any(token in status for token in ["technical failure", "timeout", "redirect error", "session expired", "failed", "error"]):
        return "technical_failure"
    if any(token in status for token in ["not booked", "abandoned", "unknown", "cancelled"]):
        return "ambiguous"
    return "ambiguous"


### Function: `label_confidence`

This block is extracted directly from `src/labels.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def label_confidence(label: str) -> float:
    return 1.0 if label in FINAL_LABELS else 0.5


### Function: `build_label_frame`

This block is extracted directly from `src/labels.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def build_label_frame(status_series: pd.Series) -> pd.DataFrame:
    labels = status_series.apply(canonicalize_status)
    return pd.DataFrame(
        {
            "raw_status": status_series.astype("string"),
            "outcome_label": labels,
            "label_confidence": labels.map(label_confidence),
            "is_ambiguous": labels.eq("ambiguous"),
        }
    )


### Function: `build_label_diagnostics`

This block is extracted directly from `src/labels.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def build_label_diagnostics(status_series: pd.Series) -> dict:
    raw_status = status_series.fillna("<<MISSING>>").astype("string")
    canonical = raw_status.apply(canonicalize_status)
    raw_counts = raw_status.value_counts(dropna=False).to_dict()
    canonical_counts = canonical.value_counts(dropna=False).to_dict()
    return {
        "raw_status_frequencies": {str(k): int(v) for k, v in raw_counts.items()},
        "canonical_label_counts": {str(k): int(v) for k, v in canonical_counts.items()},
        "unmapped_to_ambiguous_count": int((canonical == "ambiguous").sum()),
    }


### Function: `encode_target`

This block is extracted directly from `src/labels.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def encode_target(series: pd.Series) -> Tuple[pd.Series, dict]:
    mapping = {label: index for index, label in enumerate(FINAL_LABELS)}
    return series.map(mapping), mapping


## Preprocessing Pipeline

**Source file:** `src/preprocessing.py`

This module performs the corrected preprocessing flow: schema-safe field resolution, timestamp parsing, audit logging, deduplication, and leakage-free base dataset construction.

### Imports, constants, and top-level configuration

This block contains the imports, module-level constants, and shared state used by `src/preprocessing.py`.

In [ ]:
from dataclasses import dataclass
from typing import Dict, Optional, Tuple

import numpy as np
import pandas as pd

from src.config import ProjectConfig
from src.labels import build_label_diagnostics, build_label_frame
from src.reporting import build_schema_resolution_report
from src.schema import SchemaResolution, resolve_schema
from src.utils import (
    extract_domain,
    first_present,
    normalize_text,
    parse_query_params,
    safe_to_datetime,
    safe_to_numeric,
    save_csv,
    save_json,
)


### Class: `PreprocessingResult`

This block is extracted directly from `src/preprocessing.py` so the implementation can be reviewed in smaller steps.

In [ ]:
class PreprocessingResult:
    frame: pd.DataFrame
    audit: Dict[str, object]
    schema: SchemaResolution


### Function: `preprocess_raw_file`

This block is extracted directly from `src/preprocessing.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def preprocess_raw_file(config: ProjectConfig) -> PreprocessingResult:
    config.ensure_directories()
    raw = pd.read_csv(config.raw_data_path)
    result = preprocess_raw_frame(raw, config)
    save_csv(config.processed_data_path, result.frame)
    save_json(config.reports_dir / "preprocessing_audit.json", result.audit)
    save_json(
        config.reports_dir / "schema_resolution.json",
        {
            "rows": build_schema_resolution_report(result.schema),
            "missing_required": result.schema.missing_required,
            "prediction_time_assumption": config.prediction_time_assumption,
            "selected_prediction_time_raw_column": result.schema.required["prediction_time"],
            "selected_prediction_time_alias": result.schema.matched_aliases.get("prediction_time"),
        },
    )
    save_json(config.reports_dir / "label_diagnostics.json", result.audit["label_diagnostics"])
    save_json(
        config.reports_dir / "prediction_time_metadata.json",
        {
            "selected_raw_column": result.schema.required["prediction_time"],
            "matched_alias": result.schema.matched_aliases.get("prediction_time"),
            "assumption": config.prediction_time_assumption,
        },
    )
    return result


### Function: `preprocess_raw_frame`

This block is extracted directly from `src/preprocessing.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def preprocess_raw_frame(raw: pd.DataFrame, config: ProjectConfig) -> PreprocessingResult:
    schema = resolve_schema(
        list(raw.columns),
        config.column_aliases,
        config.required_logical_fields,
        config.optional_logical_fields,
    )
    if not schema.is_valid:
        raise ValueError(f"Missing required logical fields: {schema.missing_required}")

    audit = {
        "rows_initial": int(len(raw)),
        "rows_bad_timestamp": 0,
        "rows_bad_depdate": 0,
        "rows_negative_days_to_departure": 0,
        "rows_missing_required_postmap": 0,
        "rows_duplicates_removed": 0,
        "rows_unmapped_labels": 0,
        "rows_final": 0,
        "prediction_time_assumption": config.prediction_time_assumption,
    }
    audit["label_diagnostics"] = build_label_diagnostics(raw[schema.required["status"]])

    frame = pd.DataFrame()
    frame["prediction_time"] = safe_to_datetime(raw[schema.required["prediction_time"]])
    frame["departure_date"] = safe_to_datetime(raw[schema.required["departure_date"]])
    frame["origin_airport"] = normalize_text(raw[schema.required["origin"]]).str.upper()
    frame["destination_airport"] = normalize_text(raw[schema.required["destination"]]).str.upper()
    frame["airline_code"] = normalize_text(raw[schema.required["airline"]]).str.upper()
    frame["cabin_class"] = normalize_text(raw[schema.required["cabin"]]).str.upper()

    trip_col = _optional_series(raw, schema, "trip_type")
    frame["trip_type"] = normalize_text(trip_col if trip_col is not None else pd.Series(["Unknown"] * len(raw)))

    previous_page = _optional_series(raw, schema, "previous_page")
    landing_page = _optional_series(raw, schema, "landing_page")
    user_agent = _optional_series(raw, schema, "user_agent")
    inserted_on = _optional_series(raw, schema, "inserted_on")
    fdtag = _optional_series(raw, schema, "fdtag")
    flight_segments = _optional_series(raw, schema, "flight_segments")
    provider_id = _optional_series(raw, schema, "provider_id")
    price = _optional_series(raw, schema, "price")
    market = _optional_series(raw, schema, "market")
    locale = _optional_series(raw, schema, "locale")
    device = _optional_series(raw, schema, "device")
    adults = _optional_series(raw, schema, "adults")
    children = _optional_series(raw, schema, "children")
    infants = _optional_series(raw, schema, "infants")
    stops = _optional_series(raw, schema, "stops")

    frame["outcome_time"] = safe_to_datetime(inserted_on) if inserted_on is not None else pd.NaT
    frame["referrer_domain"] = (previous_page if previous_page is not None else pd.Series([None] * len(raw))).apply(extract_domain)
    frame["landing_domain"] = (landing_page if landing_page is not None else pd.Series([None] * len(raw))).apply(extract_domain)
    frame["meta_engine"] = frame["referrer_domain"].map(map_meta_engine).fillna("Direct")

    provider_key = provider_id.astype("string") if provider_id is not None else None
    if provider_key is None or provider_key.isna().all():
        provider_key = frame["landing_domain"].replace({"Unknown": pd.NA}).fillna(frame["meta_engine"])
    frame["provider_key"] = normalize_text(provider_key)

    frame["market"] = normalize_text(market if market is not None else pd.Series(["Unknown"] * len(raw)))
    frame["locale"] = normalize_text(locale if locale is not None else _extract_locale_series(user_agent, len(raw)))
    frame["device_os"] = normalize_text(device if device is not None else _extract_device_os_series(user_agent, len(raw)))

    passenger_frame = _build_passenger_frame(landing_page, adults, children, infants, len(raw))
    frame = pd.concat([frame, passenger_frame], axis=1)

    frame["stop_count"] = _build_stop_count(stops, fdtag, flight_segments, len(raw))
    frame["price_total"] = safe_to_numeric(price) if price is not None else np.nan

    label_frame = build_label_frame(raw[schema.required["status"]])
    frame = pd.concat([frame, label_frame], axis=1)

    audit["rows_bad_timestamp"] = int(frame["prediction_time"].isna().sum())
    audit["rows_bad_depdate"] = int(frame["departure_date"].isna().sum())
    audit["rows_unmapped_labels"] = int(frame["outcome_label"].eq("ambiguous").sum())

    frame = frame.dropna(subset=["prediction_time", "departure_date"]).copy()

    frame["days_to_departure"] = (frame["departure_date"] - frame["prediction_time"]).dt.days
    negative_mask = frame["days_to_departure"] < 0
    audit["rows_negative_days_to_departure"] = int(negative_mask.sum())
    frame = frame.loc[~negative_mask].copy()

    frame["prediction_time_rounded"] = frame["prediction_time"].dt.floor("min")
    frame["route"] = frame["origin_airport"] + "_" + frame["destination_airport"]

    required_postmap = [
        "prediction_time",
        "departure_date",
        "origin_airport",
        "destination_airport",
        "airline_code",
        "cabin_class",
        "outcome_label",
    ]
    missing_required_mask = frame[required_postmap].isna().any(axis=1)
    audit["rows_missing_required_postmap"] = int(missing_required_mask.sum())
    frame = frame.loc[~missing_required_mask].copy()

    dedup_key = [
        "prediction_time_rounded",
        "provider_key",
        "route",
        "airline_code",
        "cabin_class",
        "price_total",
    ]
    before_dedup = len(frame)
    frame = frame.sort_values(["prediction_time", "departure_date"]).drop_duplicates(subset=dedup_key, keep="first")
    audit["rows_duplicates_removed"] = int(before_dedup - len(frame))

    frame["unknown_provider"] = frame["provider_key"].eq("Unknown").astype(int)
    frame["missing_price"] = frame["price_total"].isna().astype(int)
    frame["stop_count"] = frame["stop_count"].fillna(0).clip(lower=0).astype(int)
    frame["is_multi_stop"] = frame["stop_count"].ge(2).astype(int)

    frame = frame.sort_values("prediction_time").reset_index(drop=True)
    audit["rows_final"] = int(len(frame))
    return PreprocessingResult(frame=frame, audit=audit, schema=schema)


### Function: `map_meta_engine`

This block is extracted directly from `src/preprocessing.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def map_meta_engine(domain: str) -> str:
    text = (domain or "").lower()
    if "skyscanner" in text:
        return "Skyscanner"
    if "google" in text:
        return "Google"
    if "kayak" in text:
        return "Kayak"
    if "momondo" in text:
        return "Momondo"
    if "tripadvisor" in text:
        return "TripAdvisor"
    if text in {"", "unknown"}:
        return "Direct"
    return "Other"


### Function: `_optional_series`

This block is extracted directly from `src/preprocessing.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def _optional_series(raw: pd.DataFrame, schema: SchemaResolution, logical_name: str) -> Optional[pd.Series]:
    column = schema.optional.get(logical_name)
    return raw[column] if column is not None else None


### Function: `_extract_device_os_series`

This block is extracted directly from `src/preprocessing.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def _extract_device_os_series(user_agent: Optional[pd.Series], length: int) -> pd.Series:
    if user_agent is None:
        return pd.Series(["Unknown"] * length)

    def parse(agent: object) -> str:
        text = str(agent).lower()
        if "android" in text:
            return "Android"
        if "iphone" in text or "ipad" in text or "ios" in text:
            return "iOS"
        if "windows" in text:
            return "Windows"
        if "mac os" in text or "macos" in text:
            return "MacOS"
        if "linux" in text:
            return "Linux"
        return "Other"

    return user_agent.apply(parse)


### Function: `_extract_locale_series`

This block is extracted directly from `src/preprocessing.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def _extract_locale_series(user_agent: Optional[pd.Series], length: int) -> pd.Series:
    if user_agent is None:
        return pd.Series(["Unknown"] * length)

    def parse(agent: object) -> str:
        text = str(agent)
        marker = "Accept-Language="
        if marker not in text:
            return "Unknown"
        suffix = text.split(marker, 1)[1]
        locale = suffix.split("&", 1)[0]
        return locale.replace("%2c", ",").replace("%3b", ";")[:32] or "Unknown"

    return user_agent.apply(parse)


### Function: `_build_passenger_frame`

This block is extracted directly from `src/preprocessing.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def _build_passenger_frame(
    landing_page: Optional[pd.Series],
    adults: Optional[pd.Series],
    children: Optional[pd.Series],
    infants: Optional[pd.Series],
    length: int,
) -> pd.DataFrame:
    if landing_page is not None:
        params = landing_page.apply(parse_query_params)
        adult_values = params.apply(lambda q: first_present(q, ["totaladults", "adults"], "1"))
        child_values = params.apply(lambda q: first_present(q, ["totalchilds", "children"], "0"))
        infant_values = params.apply(lambda q: first_present(q, ["totalinfants", "infants"], "0"))
    else:
        adult_values = adults if adults is not None else pd.Series(["1"] * length)
        child_values = children if children is not None else pd.Series(["0"] * length)
        infant_values = infants if infants is not None else pd.Series(["0"] * length)

    adults_num = safe_to_numeric(adult_values).fillna(1).clip(lower=0)
    children_num = safe_to_numeric(child_values).fillna(0).clip(lower=0)
    infants_num = safe_to_numeric(infant_values).fillna(0).clip(lower=0)

    return pd.DataFrame(
        {
            "adults": adults_num.astype(int),
            "children": children_num.astype(int),
            "infants": infants_num.astype(int),
            "passenger_count": (adults_num + children_num + infants_num).astype(int),
        }
    )


### Function: `_build_stop_count`

This block is extracted directly from `src/preprocessing.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def _build_stop_count(
    stops: Optional[pd.Series],
    fdtag: Optional[pd.Series],
    flight_segments: Optional[pd.Series],
    length: int,
) -> pd.Series:
    if stops is not None:
        numeric = safe_to_numeric(stops)
        if numeric.notna().any():
            return numeric.fillna(0)

    if fdtag is not None:
        segments = fdtag.astype("string").fillna("").str.count("~") + 1
    elif flight_segments is not None:
        segments = flight_segments.astype("string").fillna("").str.count(",") + 1
    else:
        segments = pd.Series([1] * length)

    return (segments - 1).clip(lower=0)


## Feature Engineering

**Source file:** `src/features.py`

This module builds only valid feature families: topology, temporal context, real-price features when available, and prior-only reliability/history signals.

### Imports, constants, and top-level configuration

This block contains the imports, module-level constants, and shared state used by `src/features.py`.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd


### Class: `FeatureBundle`

This block is extracted directly from `src/features.py` so the implementation can be reviewed in smaller steps.

In [ ]:
class FeatureBundle:
    frame: pd.DataFrame
    feature_groups: Dict[str, List[str]]
    categorical_features: List[str]
    numeric_features: List[str]
    inference_reference: Dict[str, object]
    feature_availability: List[Dict[str, object]]


### Function: `build_feature_bundle`

This block is extracted directly from `src/features.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def build_feature_bundle(df: pd.DataFrame, include_labels: bool = True) -> FeatureBundle:
    frame = build_base_frame(df)

    if include_labels and "outcome_label" in frame.columns:
        frame = add_history_features(frame)
        if frame["price_total"].notna().any():
            frame = add_price_features(frame)
        else:
            frame = add_price_defaults(frame)
    else:
        frame = add_snapshot_defaults(frame)

    return _package_feature_bundle(frame)


### Function: `_package_feature_bundle`

This block is extracted directly from `src/features.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def _package_feature_bundle(frame: pd.DataFrame) -> FeatureBundle:
    frame = finalize_price_features(frame.copy())

    categorical_features = [
        "trip_type",
        "cabin_class",
        "origin_airport",
        "destination_airport",
        "airline_code",
        "route",
        "airline_route",
        "provider_key",
        "provider_route",
        "provider_airline",
        "meta_engine",
        "referrer_domain",
        "market",
        "locale",
        "device_os",
        "days_to_departure_bucket",
    ]

    numeric_features = [
        "days_to_departure",
        "search_hour",
        "search_dayofweek",
        "search_month",
        "departure_month",
        "is_weekend_search",
        "passenger_count",
        "adults",
        "children",
        "infants",
        "stop_count",
        "is_multi_stop",
        "is_short_lead",
        "is_long_lead",
        "route_search_density",
        "provider_offer_density",
        "missing_price",
        "unknown_provider",
        "price_total",
        "route_day_min_price",
        "route_day_median_price",
        "price_gap_to_min",
        "price_ratio_to_median",
        "price_rank_in_route_day",
        "offers_in_route_day",
        "provider_prior_count",
        "provider_prior_rate_bookable",
        "provider_prior_rate_price_changed",
        "provider_prior_rate_unavailable",
        "provider_prior_rate_technical_failure",
        "route_prior_count",
        "route_prior_rate_bookable",
        "route_prior_rate_unavailable",
        "provider_route_prior_rate_bookable",
        "provider_route_prior_rate_unavailable",
        "airline_route_prior_rate_unavailable",
        "provider_minutes_since_prev",
        "route_minutes_since_prev",
        "provider_route_minutes_since_prev",
    ]

    for column in categorical_features:
        frame[column] = frame[column].astype("string").fillna("Unknown")
    for column in numeric_features:
        frame[column] = pd.to_numeric(frame[column], errors="coerce").fillna(0.0)

    feature_groups = {
        "topology_basic": [
            "trip_type",
            "cabin_class",
            "origin_airport",
            "destination_airport",
            "airline_code",
            "route",
            "airline_route",
            "provider_key",
            "meta_engine",
            "referrer_domain",
            "market",
            "locale",
            "device_os",
            "provider_airline",
            "passenger_count",
            "stop_count",
            "is_multi_stop",
            "unknown_provider",
        ],
        "temporal": [
            "days_to_departure",
            "search_hour",
            "search_dayofweek",
            "search_month",
            "departure_month",
            "is_weekend_search",
            "days_to_departure_bucket",
            "is_short_lead",
            "is_long_lead",
            "route_search_density",
            "provider_offer_density",
        ],
        "price": [
            "price_total",
            "route_day_min_price",
            "route_day_median_price",
            "price_gap_to_min",
            "price_ratio_to_median",
            "price_rank_in_route_day",
            "offers_in_route_day",
            "missing_price",
        ],
        "reliability_history": [
            "provider_prior_count",
            "provider_prior_rate_bookable",
            "provider_prior_rate_price_changed",
            "provider_prior_rate_unavailable",
            "provider_prior_rate_technical_failure",
            "route_prior_count",
            "route_prior_rate_bookable",
            "route_prior_rate_unavailable",
            "provider_route_prior_rate_bookable",
            "provider_route_prior_rate_unavailable",
            "airline_route_prior_rate_unavailable",
            "provider_minutes_since_prev",
            "route_minutes_since_prev",
            "provider_route_minutes_since_prev",
        ],
    }
    feature_groups["full_valid_model"] = (
        feature_groups["topology_basic"]
        + feature_groups["temporal"]
        + feature_groups["price"]
        + feature_groups["reliability_history"]
    )

    inference_reference = build_inference_reference(frame)
    feature_availability = build_feature_availability(frame, feature_groups)
    return FeatureBundle(
        frame=frame,
        feature_groups=feature_groups,
        categorical_features=categorical_features,
        numeric_features=numeric_features,
        inference_reference=inference_reference,
        feature_availability=feature_availability,
    )


### Function: `build_base_frame`

This block is extracted directly from `src/features.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def build_base_frame(df: pd.DataFrame) -> pd.DataFrame:
    frame = df.copy().sort_values("prediction_time").reset_index(drop=True)
    frame["route"] = frame["origin_airport"] + "_" + frame["destination_airport"]
    frame["airline_route"] = frame["airline_code"] + "_" + frame["route"]
    frame["provider_route"] = frame["provider_key"] + "_" + frame["route"]
    frame["provider_airline"] = frame["provider_key"] + "_" + frame["airline_code"]
    frame["route_day"] = frame["route"] + "_" + frame["departure_date"].dt.strftime("%Y-%m-%d")
    frame["search_group_proxy"] = frame["route"] + "_" + frame["trip_type"].astype("string") + "_" + frame["cabin_class"].astype("string") + "_" + frame["passenger_count"].astype("string") + "_" + frame["prediction_time"].dt.strftime("%Y-%m-%d-%H")
    frame["search_hour"] = frame["prediction_time"].dt.hour
    frame["search_dayofweek"] = frame["prediction_time"].dt.dayofweek
    frame["search_month"] = frame["prediction_time"].dt.month
    frame["departure_month"] = frame["departure_date"].dt.month
    frame["is_weekend_search"] = frame["search_dayofweek"].isin([5, 6]).astype(int)
    frame["days_to_departure_bucket"] = pd.cut(
        frame["days_to_departure"],
        bins=[-1, 1, 3, 7, 14, 30, 90, 365, np.inf],
        labels=["same_day", "1_3_days", "3_7_days", "7_14_days", "14_30_days", "30_90_days", "90_365_days", "365_plus_days"],
    ).astype("string")
    frame["is_short_lead"] = frame["days_to_departure"].le(7).astype(int)
    frame["is_long_lead"] = frame["days_to_departure"].ge(60).astype(int)
    frame["missing_price"] = frame["price_total"].isna().astype(int)
    frame["route_search_density"] = 0.0
    frame["provider_offer_density"] = 0.0
    return frame


### Function: `build_snapshot_feature_bundle`

This block is extracted directly from `src/features.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def build_snapshot_feature_bundle(target_df: pd.DataFrame, history_df: pd.DataFrame) -> FeatureBundle:
    target_frame = build_base_frame(target_df)
    target_frame = apply_history_snapshot(target_frame, history_df)
    return _package_feature_bundle(target_frame)


### Function: `add_price_features`

This block is extracted directly from `src/features.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def add_price_features(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    prior_group = frame.groupby("route_day", sort=False)["price_total"]

    frame["offers_in_route_day"] = prior_group.cumcount()
    frame["route_day_min_price"] = prior_group.transform(lambda s: s.shift(1).expanding().min())
    frame["route_day_median_price"] = prior_group.transform(lambda s: s.shift(1).expanding().median())

    global_median = float(frame["price_total"].median()) if frame["price_total"].notna().any() else 0.0
    frame["route_day_min_price"] = frame["route_day_min_price"].fillna(frame["price_total"]).fillna(global_median)
    frame["route_day_median_price"] = frame["route_day_median_price"].fillna(global_median)

    frame["price_gap_to_min"] = (frame["price_total"] - frame["route_day_min_price"]).clip(lower=0)
    denom = frame["route_day_median_price"].replace(0, np.nan)
    frame["price_ratio_to_median"] = (frame["price_total"] / denom).replace([np.inf, -np.inf], np.nan).fillna(1.0)

    def prior_rank(series: pd.Series) -> pd.Series:
        shifted = series.shift(1)
        return shifted.expanding().apply(
            lambda values: float(np.sum(values <= values.iloc[-1])) if len(values) else 0.0,
            raw=False,
        )

    frame["price_rank_in_route_day"] = prior_group.transform(prior_rank).fillna(0.0)
    return frame


### Function: `finalize_price_features`

This block is extracted directly from `src/features.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def finalize_price_features(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    frame["price_total"] = frame["price_total"].fillna(0.0)
    frame["route_day_min_price"] = frame["route_day_min_price"].fillna(frame["price_total"])
    frame["route_day_median_price"] = frame["route_day_median_price"].fillna(frame["price_total"].replace(0, np.nan)).fillna(0.0)
    frame["price_gap_to_min"] = (frame["price_total"] - frame["route_day_min_price"]).clip(lower=0)
    denom = frame["route_day_median_price"].replace(0, np.nan)
    frame["price_ratio_to_median"] = (frame["price_total"] / denom).replace([np.inf, -np.inf], np.nan).fillna(1.0)
    frame["price_rank_in_route_day"] = frame["price_rank_in_route_day"].fillna(0.0)
    frame["offers_in_route_day"] = frame["offers_in_route_day"].fillna(0.0)
    return frame


### Function: `add_history_features`

This block is extracted directly from `src/features.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def add_history_features(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    frame["is_bookable"] = frame["outcome_label"].eq("bookable").astype(int)
    frame["is_price_changed"] = frame["outcome_label"].eq("price_changed").astype(int)
    frame["is_unavailable"] = frame["outcome_label"].eq("unavailable").astype(int)
    frame["is_technical_failure"] = frame["outcome_label"].eq("technical_failure").astype(int)

    frame["provider_prior_count"] = frame.groupby("provider_key").cumcount()
    frame["route_prior_count"] = frame.groupby("route").cumcount()

    frame["provider_prior_rate_bookable"] = _prior_rate(frame, "provider_key", "is_bookable")
    frame["provider_prior_rate_price_changed"] = _prior_rate(frame, "provider_key", "is_price_changed")
    frame["provider_prior_rate_unavailable"] = _prior_rate(frame, "provider_key", "is_unavailable")
    frame["provider_prior_rate_technical_failure"] = _prior_rate(frame, "provider_key", "is_technical_failure")
    frame["route_prior_rate_bookable"] = _prior_rate(frame, "route", "is_bookable")
    frame["route_prior_rate_unavailable"] = _prior_rate(frame, "route", "is_unavailable")
    frame["provider_route_prior_rate_bookable"] = _prior_rate(frame, "provider_route", "is_bookable")
    frame["provider_route_prior_rate_unavailable"] = _prior_rate(frame, "provider_route", "is_unavailable")
    frame["airline_route_prior_rate_unavailable"] = _prior_rate(frame, "airline_route", "is_unavailable")

    frame["provider_minutes_since_prev"] = _minutes_since_previous(frame, "provider_key")
    frame["route_minutes_since_prev"] = _minutes_since_previous(frame, "route")
    frame["provider_route_minutes_since_prev"] = _minutes_since_previous(frame, "provider_route")

    global_bookable = float(frame["is_bookable"].mean())
    global_price_changed = float(frame["is_price_changed"].mean())
    global_unavailable = float(frame["is_unavailable"].mean())
    global_technical_failure = float(frame["is_technical_failure"].mean())
    time_fill = {
        "provider_minutes_since_prev": 60.0,
        "route_minutes_since_prev": 60.0,
        "provider_route_minutes_since_prev": 60.0,
    }
    frame["provider_prior_rate_bookable"] = frame["provider_prior_rate_bookable"].fillna(global_bookable)
    frame["provider_prior_rate_price_changed"] = frame["provider_prior_rate_price_changed"].fillna(global_price_changed)
    frame["provider_prior_rate_unavailable"] = frame["provider_prior_rate_unavailable"].fillna(global_unavailable)
    frame["provider_prior_rate_technical_failure"] = frame["provider_prior_rate_technical_failure"].fillna(global_technical_failure)
    frame["route_prior_rate_bookable"] = frame["route_prior_rate_bookable"].fillna(global_bookable)
    frame["route_prior_rate_unavailable"] = frame["route_prior_rate_unavailable"].fillna(global_unavailable)
    frame["provider_route_prior_rate_bookable"] = frame["provider_route_prior_rate_bookable"].fillna(global_bookable)
    frame["provider_route_prior_rate_unavailable"] = frame["provider_route_prior_rate_unavailable"].fillna(global_unavailable)
    frame["airline_route_prior_rate_unavailable"] = frame["airline_route_prior_rate_unavailable"].fillna(global_unavailable)
    for column, fill_value in time_fill.items():
        frame[column] = frame[column].fillna(fill_value)

    return frame.drop(columns=["is_bookable", "is_price_changed", "is_unavailable", "is_technical_failure"])


### Function: `add_inference_history_defaults`

This block is extracted directly from `src/features.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def add_inference_history_defaults(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    for column in [
        "provider_prior_count",
        "provider_prior_rate_bookable",
        "provider_prior_rate_price_changed",
        "provider_prior_rate_unavailable",
        "provider_prior_rate_technical_failure",
        "route_prior_count",
        "route_prior_rate_bookable",
        "route_prior_rate_unavailable",
        "provider_route_prior_rate_bookable",
        "provider_route_prior_rate_unavailable",
        "airline_route_prior_rate_unavailable",
        "provider_minutes_since_prev",
        "route_minutes_since_prev",
        "provider_route_minutes_since_prev",
    ]:
        frame[column] = 0.0
    return frame


### Function: `add_snapshot_defaults`

This block is extracted directly from `src/features.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def add_snapshot_defaults(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    for column in [
        "provider_prior_count",
        "provider_prior_rate_bookable",
        "provider_prior_rate_price_changed",
        "provider_prior_rate_unavailable",
        "provider_prior_rate_technical_failure",
        "route_prior_count",
        "route_prior_rate_bookable",
        "route_prior_rate_unavailable",
        "provider_route_prior_rate_bookable",
        "provider_route_prior_rate_unavailable",
        "airline_route_prior_rate_unavailable",
        "provider_minutes_since_prev",
        "route_minutes_since_prev",
        "provider_route_minutes_since_prev",
    ]:
        frame[column] = 0.0
    return add_price_defaults(frame)


### Function: `add_price_defaults`

This block is extracted directly from `src/features.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def add_price_defaults(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    for column in [
        "route_day_min_price",
        "route_day_median_price",
        "price_gap_to_min",
        "price_ratio_to_median",
        "price_rank_in_route_day",
        "offers_in_route_day",
    ]:
        frame[column] = 0.0
    return finalize_price_features(frame)


### Function: `apply_history_snapshot`

This block is extracted directly from `src/features.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def apply_history_snapshot(target_frame: pd.DataFrame, history_df: pd.DataFrame) -> pd.DataFrame:
    history = build_base_frame(history_df)
    history = history.sort_values("prediction_time").reset_index(drop=True)
    history["provider_prior_count"] = history.groupby("provider_key").cumcount()
    history["route_prior_count"] = history.groupby("route").cumcount()

    for label_name in ["bookable", "price_changed", "unavailable", "technical_failure"]:
        history[f"is_{label_name}"] = history["outcome_label"].eq(label_name).astype(int)

    snapshot = {
        "provider_counts": history.groupby("provider_key").size().to_dict(),
        "route_counts": history.groupby("route").size().to_dict(),
        "provider_rate_bookable": history.groupby("provider_key")["is_bookable"].mean().to_dict(),
        "provider_rate_price_changed": history.groupby("provider_key")["is_price_changed"].mean().to_dict(),
        "provider_rate_unavailable": history.groupby("provider_key")["is_unavailable"].mean().to_dict(),
        "provider_rate_technical_failure": history.groupby("provider_key")["is_technical_failure"].mean().to_dict(),
        "route_rate_bookable": history.groupby("route")["is_bookable"].mean().to_dict(),
        "route_rate_unavailable": history.groupby("route")["is_unavailable"].mean().to_dict(),
        "provider_route_rate_bookable": history.groupby("provider_route")["is_bookable"].mean().to_dict(),
        "provider_route_rate_unavailable": history.groupby("provider_route")["is_unavailable"].mean().to_dict(),
        "airline_route_rate_unavailable": history.groupby("airline_route")["is_unavailable"].mean().to_dict(),
        "provider_last_time": history.groupby("provider_key")["prediction_time"].max().to_dict(),
        "route_last_time": history.groupby("route")["prediction_time"].max().to_dict(),
        "provider_route_last_time": history.groupby("provider_route")["prediction_time"].max().to_dict(),
        "provider_first_time": history.groupby("provider_key")["prediction_time"].min().to_dict(),
        "route_first_time": history.groupby("route")["prediction_time"].min().to_dict(),
        "route_day_min_price": history.groupby("route_day")["price_total"].min().to_dict() if history["price_total"].notna().any() else {},
        "route_day_median_price": history.groupby("route_day")["price_total"].median().to_dict() if history["price_total"].notna().any() else {},
        "route_day_offer_count": history.groupby("route_day").size().to_dict(),
        "global_rates": {
            "bookable": float(history["is_bookable"].mean()) if len(history) else 0.0,
            "price_changed": float(history["is_price_changed"].mean()) if len(history) else 0.0,
            "unavailable": float(history["is_unavailable"].mean()) if len(history) else 0.0,
            "technical_failure": float(history["is_technical_failure"].mean()) if len(history) else 0.0,
        },
    }

    target = target_frame.copy()
    target["provider_prior_count"] = target["provider_key"].map(snapshot["provider_counts"]).fillna(0.0)
    target["route_prior_count"] = target["route"].map(snapshot["route_counts"]).fillna(0.0)
    target["provider_prior_rate_bookable"] = target["provider_key"].map(snapshot["provider_rate_bookable"]).fillna(snapshot["global_rates"]["bookable"])
    target["provider_prior_rate_price_changed"] = target["provider_key"].map(snapshot["provider_rate_price_changed"]).fillna(snapshot["global_rates"]["price_changed"])
    target["provider_prior_rate_unavailable"] = target["provider_key"].map(snapshot["provider_rate_unavailable"]).fillna(snapshot["global_rates"]["unavailable"])
    target["provider_prior_rate_technical_failure"] = target["provider_key"].map(snapshot["provider_rate_technical_failure"]).fillna(snapshot["global_rates"]["technical_failure"])
    target["route_prior_rate_bookable"] = target["route"].map(snapshot["route_rate_bookable"]).fillna(snapshot["global_rates"]["bookable"])
    target["route_prior_rate_unavailable"] = target["route"].map(snapshot["route_rate_unavailable"]).fillna(snapshot["global_rates"]["unavailable"])
    target["provider_route_prior_rate_bookable"] = target["provider_route"].map(snapshot["provider_route_rate_bookable"]).fillna(snapshot["global_rates"]["bookable"])
    target["provider_route_prior_rate_unavailable"] = target["provider_route"].map(snapshot["provider_route_rate_unavailable"]).fillna(snapshot["global_rates"]["unavailable"])
    target["airline_route_prior_rate_unavailable"] = target["airline_route"].map(snapshot["airline_route_rate_unavailable"]).fillna(snapshot["global_rates"]["unavailable"])
    target["provider_minutes_since_prev"] = _minutes_since_snapshot(target, "provider_key", snapshot["provider_last_time"])
    target["route_minutes_since_prev"] = _minutes_since_snapshot(target, "route", snapshot["route_last_time"])
    target["provider_route_minutes_since_prev"] = _minutes_since_snapshot(target, "provider_route", snapshot["provider_route_last_time"])
    target["route_search_density"] = _density_from_snapshot(target, "route", snapshot["route_counts"], snapshot["route_first_time"])
    target["provider_offer_density"] = _density_from_snapshot(target, "provider_key", snapshot["provider_counts"], snapshot["provider_first_time"])
    target["route_day_min_price"] = target["route_day"].map(snapshot["route_day_min_price"]).fillna(target["price_total"])
    target["route_day_median_price"] = target["route_day"].map(snapshot["route_day_median_price"]).fillna(target["price_total"])
    target["offers_in_route_day"] = target["route_day"].map(snapshot["route_day_offer_count"]).fillna(0.0)
    target["price_rank_in_route_day"] = target["offers_in_route_day"]
    return finalize_price_features(target)


### Function: `build_inference_reference`

This block is extracted directly from `src/features.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def build_inference_reference(frame: pd.DataFrame) -> Dict[str, object]:
    history_columns = [
        "provider_prior_count",
        "provider_prior_rate_bookable",
        "provider_prior_rate_price_changed",
        "provider_prior_rate_unavailable",
        "provider_prior_rate_technical_failure",
        "route_prior_count",
        "route_prior_rate_bookable",
        "route_prior_rate_unavailable",
        "provider_route_prior_rate_bookable",
        "provider_route_prior_rate_unavailable",
        "airline_route_prior_rate_unavailable",
        "provider_minutes_since_prev",
        "route_minutes_since_prev",
        "provider_route_minutes_since_prev",
    ]
    defaults = {column: float(frame[column].median()) for column in history_columns}
    return {
        "history_defaults": defaults,
        "global_price_median": float(frame["price_total"].median()) if "price_total" in frame else 0.0,
        "inference_mode_note": (
            "History-dependent features use saved batch defaults during offline inference. "
            "This is an offline approximation, not a live online feature store."
        ),
    }


### Function: `build_feature_availability`

This block is extracted directly from `src/features.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def build_feature_availability(frame: pd.DataFrame, feature_groups: Dict[str, List[str]]) -> List[Dict[str, object]]:
    family_by_feature: Dict[str, str] = {}
    for family_name, columns in feature_groups.items():
        if family_name == "full_valid_model":
            continue
        for column in columns:
            family_by_feature[column] = family_name

    history_features = set(feature_groups["reliability_history"])
    final_model_features = set(feature_groups["full_valid_model"])
    rows = []
    for feature_name in sorted(final_model_features):
        rows.append(
            {
                "feature_name": feature_name,
                "feature_family": family_by_feature.get(feature_name, "unknown"),
                "available_at_prediction_time": True,
                "derived_from_prior_only_history": feature_name in history_features,
                "used_in_final_model": True,
                "non_null_fraction": float(frame[feature_name].notna().mean()) if feature_name in frame else 0.0,
            }
        )
    return rows


### Function: `_prior_rate`

This block is extracted directly from `src/features.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def _prior_rate(frame: pd.DataFrame, key: str, indicator: str) -> pd.Series:
    group = frame.groupby(key)[indicator]
    cumulative = group.cumsum() - frame[indicator]
    prior_count = frame.groupby(key).cumcount().replace(0, np.nan)
    return cumulative / prior_count


### Function: `_minutes_since_previous`

This block is extracted directly from `src/features.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def _minutes_since_previous(frame: pd.DataFrame, key: str) -> pd.Series:
    previous = frame.groupby(key)["prediction_time"].shift(1)
    return (frame["prediction_time"] - previous).dt.total_seconds() / 60.0


### Function: `_count_per_time`

This block is extracted directly from `src/features.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def _count_per_time(frame: pd.DataFrame, key: str) -> pd.Series:
    elapsed_days = (
        frame["prediction_time"] - frame.groupby(key)["prediction_time"].transform("min")
    ).dt.total_seconds() / 86400.0
    return (frame.groupby(key).cumcount() + 1) / elapsed_days.clip(lower=1.0)


### Function: `_minutes_since_snapshot`

This block is extracted directly from `src/features.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def _minutes_since_snapshot(target: pd.DataFrame, key: str, last_time_map: Dict[str, pd.Timestamp]) -> pd.Series:
    previous_time = target[key].map(last_time_map)
    minutes = (target["prediction_time"] - pd.to_datetime(previous_time)).dt.total_seconds() / 60.0
    return minutes.fillna(60.0).clip(lower=0.0)


### Function: `_density_from_snapshot`

This block is extracted directly from `src/features.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def _density_from_snapshot(target: pd.DataFrame, key: str, count_map: Dict[str, float], first_time_map: Dict[str, pd.Timestamp]) -> pd.Series:
    counts = target[key].map(count_map).fillna(0.0) + 1.0
    first_times = pd.to_datetime(target[key].map(first_time_map))
    elapsed_days = (target["prediction_time"] - first_times).dt.total_seconds() / 86400.0
    return counts / elapsed_days.fillna(1.0).clip(lower=1.0)


## Temporal Splitting

**Source file:** `src/split.py`

Chronological train, validation, and test splitting plus rolling backtest window generation live here.

### Imports, constants, and top-level configuration

This block contains the imports, module-level constants, and shared state used by `src/split.py`.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List

import pandas as pd


### Class: `TemporalSplit`

This block is extracted directly from `src/split.py` so the implementation can be reviewed in smaller steps.

In [ ]:
class TemporalSplit:
    train: pd.DataFrame
    validation: pd.DataFrame
    test: pd.DataFrame
    metadata: Dict[str, object]


### Function: `temporal_train_validation_test_split`

This block is extracted directly from `src/split.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def temporal_train_validation_test_split(
    frame: pd.DataFrame,
    train_fraction: float,
    validation_fraction: float,
    test_fraction: float,
) -> TemporalSplit:
    if round(train_fraction + validation_fraction + test_fraction, 6) != 1.0:
        raise ValueError("Train/validation/test fractions must sum to 1.0")

    ordered = frame.sort_values("prediction_time").reset_index(drop=True)
    n_rows = len(ordered)
    train_end = int(n_rows * train_fraction)
    validation_end = int(n_rows * (train_fraction + validation_fraction))

    train = ordered.iloc[:train_end].copy()
    validation = ordered.iloc[train_end:validation_end].copy()
    test = ordered.iloc[validation_end:].copy()

    metadata = {
        "n_rows": n_rows,
        "train_rows": len(train),
        "validation_rows": len(validation),
        "test_rows": len(test),
        "train_start": str(train["prediction_time"].min()) if not train.empty else None,
        "train_end": str(train["prediction_time"].max()) if not train.empty else None,
        "validation_start": str(validation["prediction_time"].min()) if not validation.empty else None,
        "validation_end": str(validation["prediction_time"].max()) if not validation.empty else None,
        "test_start": str(test["prediction_time"].min()) if not test.empty else None,
        "test_end": str(test["prediction_time"].max()) if not test.empty else None,
        "class_distribution": {
            "train": train["outcome_label"].value_counts().to_dict() if "outcome_label" in train else {},
            "validation": validation["outcome_label"].value_counts().to_dict() if "outcome_label" in validation else {},
            "test": test["outcome_label"].value_counts().to_dict() if "outcome_label" in test else {},
        },
    }
    return TemporalSplit(train=train, validation=validation, test=test, metadata=metadata)


### Function: `rolling_backtest_splits`

This block is extracted directly from `src/split.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def rolling_backtest_splits(frame: pd.DataFrame, n_windows: int = 3) -> List[Dict[str, pd.DataFrame]]:
    ordered = frame.sort_values("prediction_time").reset_index(drop=True)
    n_rows = len(ordered)
    if n_windows < 1:
        raise ValueError("n_windows must be >= 1")

    step = n_rows // (n_windows + 2)
    windows: List[Dict[str, pd.DataFrame]] = []

    for idx in range(n_windows):
        train_end = step * (idx + 2)
        test_end = min(step * (idx + 3), n_rows)
        train = ordered.iloc[:train_end].copy()
        test = ordered.iloc[train_end:test_end].copy()
        if train.empty or test.empty:
            continue
        windows.append(
            {
                "window_id": idx + 1,
                "train": train,
                "test": test,
                "train_start": str(train["prediction_time"].min()),
                "train_end": str(train["prediction_time"].max()),
                "test_start": str(test["prediction_time"].min()),
                "test_end": str(test["prediction_time"].max()),
            }
        )
    return windows


## Evaluation Metrics

**Source file:** `src/metrics.py`

This module implements the required research metrics including macro F1, weighted F1, balanced accuracy, log loss, multiclass Brier, confusion matrix, and classwise ECE.

### Imports, constants, and top-level configuration

This block contains the imports, module-level constants, and shared state used by `src/metrics.py`.

In [ ]:
from typing import Dict, List

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
)


### Function: `multiclass_brier_score`

This block is extracted directly from `src/metrics.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def multiclass_brier_score(y_true: np.ndarray, y_prob: np.ndarray, labels: List[int]) -> float:
    one_hot = np.eye(len(labels))[y_true]
    return float(np.mean(np.sum((y_prob - one_hot) ** 2, axis=1)))


### Function: `expected_calibration_error`

This block is extracted directly from `src/metrics.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def expected_calibration_error(y_true_binary: np.ndarray, y_prob_binary: np.ndarray, n_bins: int = 10) -> float:
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for idx in range(n_bins):
        mask = (y_prob_binary >= bins[idx]) & (y_prob_binary < bins[idx + 1] if idx < n_bins - 1 else y_prob_binary <= bins[idx + 1])
        if not np.any(mask):
            continue
        avg_conf = np.mean(y_prob_binary[mask])
        avg_acc = np.mean(y_true_binary[mask])
        ece += (np.sum(mask) / len(y_prob_binary)) * abs(avg_conf - avg_acc)
    return float(ece)


### Function: `classwise_ece`

This block is extracted directly from `src/metrics.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def classwise_ece(y_true: np.ndarray, y_prob: np.ndarray, labels: List[str]) -> Dict[str, float]:
    metrics = {}
    for idx, label in enumerate(labels):
        binary_true = (y_true == idx).astype(int)
        metrics[label] = expected_calibration_error(binary_true, y_prob[:, idx])
    return metrics


### Function: `reliability_table`

This block is extracted directly from `src/metrics.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def reliability_table(y_true: np.ndarray, y_prob: np.ndarray, labels: List[str], n_bins: int = 10) -> Dict[str, List[Dict[str, float]]]:
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    output: Dict[str, List[Dict[str, float]]] = {}
    for idx, label in enumerate(labels):
        binary_true = (y_true == idx).astype(int)
        class_rows = []
        for bin_index in range(n_bins):
            lower = bins[bin_index]
            upper = bins[bin_index + 1]
            mask = (y_prob[:, idx] >= lower) & (y_prob[:, idx] < upper if bin_index < n_bins - 1 else y_prob[:, idx] <= upper)
            if not np.any(mask):
                continue
            class_rows.append(
                {
                    "bin_lower": float(lower),
                    "bin_upper": float(upper),
                    "avg_confidence": float(np.mean(y_prob[mask, idx])),
                    "avg_accuracy": float(np.mean(binary_true[mask])),
                    "count": int(np.sum(mask)),
                }
            )
        output[label] = class_rows
    return output


### Function: `classification_metrics`

This block is extracted directly from `src/metrics.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def classification_metrics(y_true: np.ndarray, y_prob: np.ndarray, labels: List[str]) -> Dict[str, object]:
    predicted = np.argmax(y_prob, axis=1)
    numeric_labels = list(range(len(labels)))
    metrics = {
        "accuracy": float(accuracy_score(y_true, predicted)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, predicted)),
        "macro_f1": float(f1_score(y_true, predicted, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, predicted, average="weighted", zero_division=0)),
        "log_loss": float(log_loss(y_true, y_prob, labels=numeric_labels)),
        "multiclass_brier": multiclass_brier_score(y_true, y_prob, numeric_labels),
        "confusion_matrix": confusion_matrix(y_true, predicted, labels=numeric_labels).tolist(),
        "classification_report": classification_report(
            y_true,
            predicted,
            labels=numeric_labels,
            target_names=labels,
            zero_division=0,
            output_dict=True,
        ),
        "classwise_ece": classwise_ece(y_true, y_prob, labels),
        "reliability_table": reliability_table(y_true, y_prob, labels),
    }
    metrics["ece_macro"] = float(np.mean(list(metrics["classwise_ece"].values())))
    return metrics


## Model Comparison

**Source file:** `src/models.py`

This module defines the candidate model families and the transparent model selection rule used on the validation split.

### Imports, constants, and top-level configuration

This block contains the imports, module-level constants, and shared state used by `src/models.py`.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.metrics import classification_metrics


### Class: `ModelResult`

This block is extracted directly from `src/models.py` so the implementation can be reviewed in smaller steps.

In [ ]:
class ModelResult:
    name: str
    estimator: object
    validation_probabilities: np.ndarray
    validation_metrics: Dict[str, object]
    selection_score: float
    tuning_metadata: Dict[str, object]


### Function: `compare_models`

This block is extracted directly from `src/models.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def compare_models(
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    X_validation: pd.DataFrame,
    y_validation: np.ndarray,
    labels: List[str],
    categorical_features: List[str],
    numeric_features: List[str],
    alpha: float,
    random_state: int,
) -> List[ModelResult]:
    numeric_labels = list(range(len(labels)))
    candidates = [
        _tune_logistic_regression(
            X_train,
            y_train,
            X_validation,
            y_validation,
            labels,
            categorical_features,
            numeric_features,
            alpha,
            random_state,
        ),
        _fit_random_forest(
            X_train,
            y_train,
            X_validation,
            y_validation,
            labels,
            categorical_features,
            numeric_features,
            alpha,
            random_state,
        ),
        _tune_catboost(
            X_train,
            y_train,
            X_validation,
            y_validation,
            labels,
            categorical_features,
            alpha,
            random_state,
        ),
    ]
    candidates.sort(key=lambda item: item.selection_score, reverse=True)
    return candidates


### Function: `build_candidates`

This block is extracted directly from `src/models.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def build_candidates(categorical_features: List[str], numeric_features: List[str], random_state: int) -> Dict[str, object]:
    return {
        "logistic_regression": _build_logistic_pipeline(categorical_features, numeric_features, c_value=1.0, random_state=random_state),
        "random_forest": _build_random_forest_pipeline(categorical_features, numeric_features, random_state=random_state),
        "catboost": _build_catboost(depth=8, learning_rate=0.06, iterations=300, random_state=random_state),
    }


### Function: `predict_proba_aligned`

This block is extracted directly from `src/models.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def predict_proba_aligned(estimator, X: pd.DataFrame, class_labels: List[int]) -> np.ndarray:
    raw_probabilities = estimator.predict_proba(X)
    estimator_classes = getattr(estimator, "classes_", None)
    if estimator_classes is None and hasattr(estimator, "named_steps"):
        classifier = estimator.named_steps.get("classifier")
        estimator_classes = getattr(classifier, "classes_", None)
    if estimator_classes is None:
        return raw_probabilities

    aligned = np.zeros((len(X), len(class_labels)), dtype=float)
    class_to_position = {int(label): idx for idx, label in enumerate(class_labels)}
    for source_idx, class_value in enumerate(estimator_classes):
        aligned[:, class_to_position[int(class_value)]] = raw_probabilities[:, source_idx]
    row_sums = aligned.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0.0] = 1.0
    return aligned / row_sums


### Function: `_tune_logistic_regression`

This block is extracted directly from `src/models.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def _tune_logistic_regression(
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    X_validation: pd.DataFrame,
    y_validation: np.ndarray,
    labels: List[str],
    categorical_features: List[str],
    numeric_features: List[str],
    alpha: float,
    random_state: int,
) -> ModelResult:
    trials = []
    for c_value in [0.5, 1.0, 2.0]:
        estimator = _build_logistic_pipeline(categorical_features, numeric_features, c_value=c_value, random_state=random_state)
        estimator.fit(X_train, y_train)
        probabilities = predict_proba_aligned(estimator, X_validation, list(range(len(labels))))
        metrics = classification_metrics(y_validation, probabilities, labels)
        score = metrics["macro_f1"] - alpha * metrics["log_loss"]
        trials.append((score, c_value, estimator, probabilities, metrics))

    best_score, best_c, best_estimator, best_probabilities, best_metrics = max(trials, key=lambda item: item[0])
    return ModelResult(
        name="logistic_regression",
        estimator=best_estimator,
        validation_probabilities=best_probabilities,
        validation_metrics=best_metrics,
        selection_score=float(best_score),
        tuning_metadata={"best_C": best_c, "trial_count": len(trials)},
    )


### Function: `_fit_random_forest`

This block is extracted directly from `src/models.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def _fit_random_forest(
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    X_validation: pd.DataFrame,
    y_validation: np.ndarray,
    labels: List[str],
    categorical_features: List[str],
    numeric_features: List[str],
    alpha: float,
    random_state: int,
) -> ModelResult:
    estimator = _build_random_forest_pipeline(categorical_features, numeric_features, random_state=random_state)
    estimator.fit(X_train, y_train)
    probabilities = predict_proba_aligned(estimator, X_validation, list(range(len(labels))))
    metrics = classification_metrics(y_validation, probabilities, labels)
    score = metrics["macro_f1"] - alpha * metrics["log_loss"]
    return ModelResult(
        name="random_forest",
        estimator=estimator,
        validation_probabilities=probabilities,
        validation_metrics=metrics,
        selection_score=float(score),
        tuning_metadata={"strategy": "fixed_reasonable_defaults"},
    )


### Function: `_tune_catboost`

This block is extracted directly from `src/models.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def _tune_catboost(
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    X_validation: pd.DataFrame,
    y_validation: np.ndarray,
    labels: List[str],
    categorical_features: List[str],
    alpha: float,
    random_state: int,
) -> ModelResult:
    trials = []
    grid = [
        {"depth": 6, "learning_rate": 0.05, "iterations": 250},
        {"depth": 8, "learning_rate": 0.05, "iterations": 250},
        {"depth": 6, "learning_rate": 0.08, "iterations": 350},
        {"depth": 8, "learning_rate": 0.08, "iterations": 350},
    ]
    for params in grid:
        estimator = _build_catboost(random_state=random_state, **params)
        estimator.fit(X_train, y_train, cat_features=categorical_features, eval_set=(X_validation, y_validation))
        probabilities = predict_proba_aligned(estimator, X_validation, list(range(len(labels))))
        metrics = classification_metrics(y_validation, probabilities, labels)
        score = metrics["macro_f1"] - alpha * metrics["log_loss"]
        trials.append((score, params, estimator, probabilities, metrics))

    best_score, best_params, best_estimator, best_probabilities, best_metrics = max(trials, key=lambda item: item[0])
    return ModelResult(
        name="catboost",
        estimator=best_estimator,
        validation_probabilities=best_probabilities,
        validation_metrics=best_metrics,
        selection_score=float(best_score),
        tuning_metadata={"best_params": best_params, "trial_count": len(trials)},
    )


### Function: `_build_logistic_pipeline`

This block is extracted directly from `src/models.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def _build_logistic_pipeline(categorical_features: List[str], numeric_features: List[str], c_value: float, random_state: int) -> Pipeline:
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "categorical",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("encoder", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                categorical_features,
            ),
            (
                "numeric",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_features,
            ),
        ]
    )
    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "classifier",
                LogisticRegression(
                    max_iter=1000,
                    solver="saga",
                    class_weight="balanced",
                    C=c_value,
                    random_state=random_state,
                ),
            ),
        ]
    )


### Function: `_build_random_forest_pipeline`

This block is extracted directly from `src/models.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def _build_random_forest_pipeline(categorical_features: List[str], numeric_features: List[str], random_state: int) -> Pipeline:
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "categorical",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("encoder", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                categorical_features,
            ),
            ("numeric", SimpleImputer(strategy="median"), numeric_features),
        ]
    )
    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "classifier",
                RandomForestClassifier(
                    n_estimators=250,
                    max_depth=18,
                    min_samples_leaf=2,
                    class_weight="balanced_subsample",
                    n_jobs=-1,
                    random_state=random_state,
                ),
            ),
        ]
    )


### Function: `_build_catboost`

This block is extracted directly from `src/models.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def _build_catboost(depth: int, learning_rate: float, iterations: int, random_state: int) -> CatBoostClassifier:
    return CatBoostClassifier(
        iterations=iterations,
        learning_rate=learning_rate,
        depth=depth,
        loss_function="MultiClass",
        auto_class_weights="Balanced",
        verbose=0,
        random_seed=random_state,
    )


## Calibration Comparison

**Source file:** `src/calibration.py`

This module implements the required calibration strategies: no calibration, temperature scaling, one-vs-rest isotonic, and multinomial logistic calibration.

### Imports, constants, and top-level configuration

This block contains the imports, module-level constants, and shared state used by `src/calibration.py`.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List

import numpy as np
from scipy.optimize import minimize
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression

from src.metrics import classification_metrics


### Class: `IdentityCalibrator`

This block is extracted directly from `src/calibration.py` so the implementation can be reviewed in smaller steps.

In [ ]:
class IdentityCalibrator:
    name = "none"

    def fit(self, probabilities: np.ndarray, y_true: np.ndarray) -> "IdentityCalibrator":
        self.classes_ = np.arange(probabilities.shape[1])
        return self

    def predict_proba(self, probabilities: np.ndarray) -> np.ndarray:
        return probabilities


### Class: `TemperatureScalingCalibrator`

This block is extracted directly from `src/calibration.py` so the implementation can be reviewed in smaller steps.

In [ ]:
class TemperatureScalingCalibrator:
    name = "temperature_scaling"

    def __init__(self) -> None:
        self.temperature = 1.0

    def fit(self, probabilities: np.ndarray, y_true: np.ndarray) -> "TemperatureScalingCalibrator":
        self.classes_ = np.arange(probabilities.shape[1])
        logits = np.log(np.clip(probabilities, 1e-12, 1.0))

        def objective(temp: np.ndarray) -> float:
            scaled = _softmax(logits / temp[0])
            metrics = classification_metrics(y_true, scaled, [str(i) for i in range(probabilities.shape[1])])
            return metrics["log_loss"]

        result = minimize(objective, x0=np.array([1.0]), bounds=[(0.05, 10.0)])
        self.temperature = float(result.x[0])
        return self

    def predict_proba(self, probabilities: np.ndarray) -> np.ndarray:
        logits = np.log(np.clip(probabilities, 1e-12, 1.0))
        return _softmax(logits / self.temperature)


### Class: `OVRIsotonicCalibrator`

This block is extracted directly from `src/calibration.py` so the implementation can be reviewed in smaller steps.

In [ ]:
class OVRIsotonicCalibrator:
    name = "ovr_isotonic"

    def __init__(self) -> None:
        self.models: List[IsotonicRegression] = []

    def fit(self, probabilities: np.ndarray, y_true: np.ndarray) -> "OVRIsotonicCalibrator":
        self.classes_ = np.arange(probabilities.shape[1])
        self.models = []
        for class_index in range(probabilities.shape[1]):
            isotonic = IsotonicRegression(out_of_bounds="clip")
            binary_target = (y_true == class_index).astype(int)
            isotonic.fit(probabilities[:, class_index], binary_target)
            self.models.append(isotonic)
        return self

    def predict_proba(self, probabilities: np.ndarray) -> np.ndarray:
        calibrated = np.zeros_like(probabilities)
        for class_index, isotonic in enumerate(self.models):
            calibrated[:, class_index] = isotonic.transform(probabilities[:, class_index])
        calibrated = np.clip(calibrated, 1e-9, 1.0)
        return calibrated / calibrated.sum(axis=1, keepdims=True)


### Class: `MultinomialLogisticCalibrator`

This block is extracted directly from `src/calibration.py` so the implementation can be reviewed in smaller steps.

In [ ]:
class MultinomialLogisticCalibrator:
    name = "multinomial_logistic"

    def __init__(self) -> None:
        self.model = LogisticRegression(
            solver="lbfgs",
            max_iter=500,
        )

    def fit(self, probabilities: np.ndarray, y_true: np.ndarray) -> "MultinomialLogisticCalibrator":
        features = np.log(np.clip(probabilities, 1e-12, 1.0))
        self.model.fit(features, y_true)
        self.classes_ = self.model.classes_
        return self

    def predict_proba(self, probabilities: np.ndarray) -> np.ndarray:
        features = np.log(np.clip(probabilities, 1e-12, 1.0))
        return self.model.predict_proba(features)


### Class: `CalibrationResult`

This block is extracted directly from `src/calibration.py` so the implementation can be reviewed in smaller steps.

In [ ]:
class CalibrationResult:
    name: str
    calibrator: object
    validation_metrics: Dict[str, object]
    selection_score: float


### Function: `compare_calibrators`

This block is extracted directly from `src/calibration.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def compare_calibrators(
    validation_probabilities: np.ndarray,
    y_validation: np.ndarray,
    labels: List[str],
) -> List[CalibrationResult]:
    calibrators = [
        IdentityCalibrator(),
        TemperatureScalingCalibrator(),
        OVRIsotonicCalibrator(),
        MultinomialLogisticCalibrator(),
    ]
    results: List[CalibrationResult] = []

    for calibrator in calibrators:
        calibrator.fit(validation_probabilities, y_validation)
        calibrated = align_calibrated_probabilities(calibrator, validation_probabilities, list(range(len(labels))))
        metrics = classification_metrics(y_validation, calibrated, labels)
        selection_score = metrics["macro_f1"] - 0.05 * metrics["log_loss"] - 0.10 * metrics["ece_macro"]
        results.append(
            CalibrationResult(
                name=calibrator.name,
                calibrator=calibrator,
                validation_metrics=metrics,
                selection_score=float(selection_score),
            )
        )

    results.sort(key=lambda item: item.selection_score, reverse=True)
    return results


### Function: `align_calibrated_probabilities`

This block is extracted directly from `src/calibration.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def align_calibrated_probabilities(calibrator, probabilities: np.ndarray, class_labels: List[int]) -> np.ndarray:
    raw_probabilities = calibrator.predict_proba(probabilities)
    calibrator_classes = getattr(calibrator, "classes_", None)
    if calibrator_classes is None:
        return raw_probabilities

    aligned = np.zeros((len(probabilities), len(class_labels)), dtype=float)
    class_to_position = {int(label): idx for idx, label in enumerate(class_labels)}
    for source_idx, class_value in enumerate(calibrator_classes):
        aligned[:, class_to_position[int(class_value)]] = raw_probabilities[:, source_idx]
    row_sums = aligned.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0.0] = 1.0
    return aligned / row_sums


### Function: `_softmax`

This block is extracted directly from `src/calibration.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def _softmax(values: np.ndarray) -> np.ndarray:
    shifted = values - values.max(axis=1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum(axis=1, keepdims=True)


## Evaluation Helpers

**Source file:** `src/evaluation.py`

Small reporting helpers for converting metric dictionaries into comparison tables that can be saved as report artifacts.

### Imports, constants, and top-level configuration

This block contains the imports, module-level constants, and shared state used by `src/evaluation.py`.

In [ ]:
from typing import Dict, List

import numpy as np
import pandas as pd

from src.metrics import classification_metrics


### Function: `evaluate_predictions`

This block is extracted directly from `src/evaluation.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def evaluate_predictions(y_true: np.ndarray, probabilities: np.ndarray, labels: List[str]) -> Dict[str, object]:
    return classification_metrics(y_true, probabilities, labels)


### Function: `comparison_table`

This block is extracted directly from `src/evaluation.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def comparison_table(rows: List[Dict[str, object]]) -> pd.DataFrame:
    frame = pd.DataFrame(rows)
    preferred_columns = [
        "name",
        "selection_score",
        "accuracy",
        "balanced_accuracy",
        "macro_f1",
        "weighted_f1",
        "log_loss",
        "multiclass_brier",
        "ece_macro",
    ]
    columns = [column for column in preferred_columns if column in frame.columns]
    return frame[columns].sort_values("selection_score", ascending=False).reset_index(drop=True)


## Reporting And Diagnostics

**Source file:** `src/reporting.py`

This module generates dissertation-facing diagnostics such as subgroup performance, calibration parity, and schema resolution reporting.

### Imports, constants, and top-level configuration

This block contains the imports, module-level constants, and shared state used by `src/reporting.py`.

In [ ]:
from typing import Dict, List

import numpy as np
import pandas as pd

from src.labels import FINAL_LABELS
from src.metrics import classwise_ece


### Function: `build_schema_resolution_report`

This block is extracted directly from `src/reporting.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def build_schema_resolution_report(schema_resolution, prediction_time_logical_name: str = "prediction_time") -> List[Dict[str, object]]:
    rows: List[Dict[str, object]] = []
    for logical_name, resolved in schema_resolution.required.items():
        rows.append(
            {
                "logical_field": logical_name,
                "resolved_raw_column": resolved,
                "matched_alias": schema_resolution.matched_aliases.get(logical_name),
                "is_required": True,
                "selected_as_prediction_time": logical_name == prediction_time_logical_name,
            }
        )
    for logical_name, resolved in schema_resolution.optional.items():
        rows.append(
            {
                "logical_field": logical_name,
                "resolved_raw_column": resolved,
                "matched_alias": schema_resolution.matched_aliases.get(logical_name),
                "is_required": False,
                "selected_as_prediction_time": False,
            }
        )
    return rows


### Function: `subgroup_metrics`

This block is extracted directly from `src/reporting.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def subgroup_metrics(
    frame: pd.DataFrame,
    y_true: np.ndarray,
    y_prob: np.ndarray,
    group_column: str,
    labels: List[str] | None = None,
    top_n: int = 20,
    min_rows: int = 100,
) -> List[Dict[str, object]]:
    labels = labels or FINAL_LABELS
    if group_column not in frame.columns:
        return []

    enriched = frame[[group_column]].copy()
    enriched["y_true"] = y_true
    enriched["predicted"] = np.argmax(y_prob, axis=1)
    counts = enriched[group_column].value_counts().head(top_n)

    rows: List[Dict[str, object]] = []
    for group_value, count in counts.items():
        if count < min_rows:
            continue
        mask = enriched[group_column] == group_value
        group_true = y_true[mask.to_numpy()]
        group_prob = y_prob[mask.to_numpy()]
        predicted = np.argmax(group_prob, axis=1)

        balanced_accuracy = _balanced_accuracy(group_true, predicted)
        macro_f1 = _macro_f1(group_true, predicted, len(labels))
        logloss = _safe_log_loss(group_true, group_prob)
        ece = classwise_ece(group_true, group_prob, labels)
        rows.append(
            {
                "group_column": group_column,
                "group_value": str(group_value),
                "row_count": int(count),
                "macro_f1": float(macro_f1),
                "balanced_accuracy": float(balanced_accuracy),
                "log_loss": float(logloss),
                "classwise_ece": ece,
                "ece_macro": float(np.mean(list(ece.values()))),
            }
        )
    return rows


### Function: `calibration_parity_report`

This block is extracted directly from `src/reporting.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def calibration_parity_report(
    frame: pd.DataFrame,
    y_true: np.ndarray,
    y_prob: np.ndarray,
    subgroup_columns: List[str],
    labels: List[str] | None = None,
) -> Dict[str, List[Dict[str, object]]]:
    labels = labels or FINAL_LABELS
    report: Dict[str, List[Dict[str, object]]] = {}
    for column in subgroup_columns:
        report[column] = subgroup_metrics(frame, y_true, y_prob, column, labels=labels)
    return report


### Function: `_balanced_accuracy`

This block is extracted directly from `src/reporting.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def _balanced_accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    per_class = []
    for label in np.unique(y_true):
        mask = y_true == label
        if np.sum(mask) == 0:
            continue
        per_class.append(float(np.mean(y_pred[mask] == y_true[mask])))
    return float(np.mean(per_class)) if per_class else 0.0


### Function: `_macro_f1`

This block is extracted directly from `src/reporting.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def _macro_f1(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int) -> float:
    scores = []
    for label in range(n_classes):
        tp = np.sum((y_true == label) & (y_pred == label))
        fp = np.sum((y_true != label) & (y_pred == label))
        fn = np.sum((y_true == label) & (y_pred != label))
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        if precision + recall == 0:
            scores.append(0.0)
        else:
            scores.append(2 * precision * recall / (precision + recall))
    return float(np.mean(scores))


### Function: `_safe_log_loss`

This block is extracted directly from `src/reporting.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def _safe_log_loss(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    one_hot = np.eye(y_prob.shape[1])[y_true]
    clipped = np.clip(y_prob, 1e-12, 1.0)
    return float(-np.mean(np.sum(one_hot * np.log(clipped), axis=1)))


## Operational Policy Simulation

**Source file:** `src/simulation.py`

This module contains the suppression policy simulation and the reranking proxy used to connect calibrated probabilities to metasearch decisions.

### Imports, constants, and top-level configuration

This block contains the imports, module-level constants, and shared state used by `src/simulation.py`.

In [ ]:
from typing import Dict, List

import numpy as np
import pandas as pd


### Function: `suppression_policy_simulation`

This block is extracted directly from `src/simulation.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def suppression_policy_simulation(frame: pd.DataFrame, probabilities: np.ndarray, labels: List[str], thresholds: List[float]) -> List[Dict[str, object]]:
    enriched = frame.copy()
    enriched["risk_unbookable"] = _risk_unbookable(probabilities, labels)

    rows: List[Dict[str, object]] = []
    for threshold in thresholds:
        suppressed_mask = enriched["risk_unbookable"] >= threshold
        retained_mask = ~suppressed_mask
        rows.append(
            {
                "threshold": threshold,
                "suppressed_rows": int(suppressed_mask.sum()),
                "retained_rows": int(retained_mask.sum()),
                "suppressed_true_bookable": int(((enriched["outcome_label"] == "bookable") & suppressed_mask).sum()),
                "suppressed_true_unbookable": int(((enriched["outcome_label"] != "bookable") & suppressed_mask).sum()),
                "retained_true_bookable": int(((enriched["outcome_label"] == "bookable") & retained_mask).sum()),
                "retained_true_unbookable": int(((enriched["outcome_label"] != "bookable") & retained_mask).sum()),
                "retained_bookable_rate": float((enriched.loc[retained_mask, "outcome_label"] == "bookable").mean()) if retained_mask.any() else 0.0,
            }
        )
    return rows


### Function: `reranking_proxy_simulation`

This block is extracted directly from `src/simulation.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def reranking_proxy_simulation(frame: pd.DataFrame, probabilities: np.ndarray, labels: List[str]) -> Dict[str, object]:
    enriched = frame.copy()
    bookable_index = labels.index("bookable")
    enriched["p_bookable"] = probabilities[:, bookable_index]
    enriched["risk_unbookable"] = _risk_unbookable(probabilities, labels)

    if "price_total" in enriched and enriched["price_total"].notna().any():
        price_penalty = enriched["price_ratio_to_median"].clip(lower=0.5, upper=2.0)
        baseline = (
            enriched.sort_values(["search_group_proxy", "price_total", "prediction_time"], ascending=[True, True, True])
            .groupby("search_group_proxy", as_index=False)
            .first()
        )
        baseline_name = "cheapest_offer_baseline"
    else:
        price_penalty = 1.0
        baseline = (
            enriched.sort_values(["search_group_proxy", "prediction_time"], ascending=[True, True])
            .groupby("search_group_proxy", as_index=False)
            .first()
        )
        baseline_name = "first_seen_offer_baseline"

    enriched["ml_rank_score"] = enriched["p_bookable"] - 0.35 * enriched["risk_unbookable"] - 0.05 * price_penalty
    ml_top = (
        enriched.sort_values(["search_group_proxy", "ml_rank_score"], ascending=[True, False])
        .groupby("search_group_proxy", as_index=False)
        .first()
    )

    return {
        "grouping_key": "search_group_proxy",
        "baseline_name": baseline_name,
        "num_groups": int(enriched["search_group_proxy"].nunique()),
        "baseline_top1_bookable_rate": float((baseline["outcome_label"] == "bookable").mean()),
        "ml_top1_bookable_rate": float((ml_top["outcome_label"] == "bookable").mean()),
        "baseline_top1_unavailable_rate": float((baseline["outcome_label"] == "unavailable").mean()),
        "ml_top1_unavailable_rate": float((ml_top["outcome_label"] == "unavailable").mean()),
        "baseline_top1_technical_failure_rate": float((baseline["outcome_label"] == "technical_failure").mean()),
        "ml_top1_technical_failure_rate": float((ml_top["outcome_label"] == "technical_failure").mean()),
    }


### Function: `_risk_unbookable`

This block is extracted directly from `src/simulation.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def _risk_unbookable(probabilities: np.ndarray, labels: List[str]) -> np.ndarray:
    risk_columns = [
        labels.index("price_changed"),
        labels.index("unavailable"),
        labels.index("technical_failure"),
    ]
    return probabilities[:, risk_columns].sum(axis=1)


## Main Training Pipeline

**Source file:** `src/train.py`

This is the final end-to-end training entry point. It orchestrates preprocessing, feature construction, temporal splitting, model comparison, calibration selection, evaluation, ablation studies, rolling backtests, policy simulation, and artifact saving.

### Imports, constants, and top-level configuration

This block contains the imports, module-level constants, and shared state used by `src/train.py`.

In [ ]:
import json
from typing import Dict, List

import joblib
import pandas as pd

from src.calibration import align_calibrated_probabilities, compare_calibrators
from src.config import ProjectConfig
from src.evaluation import comparison_table, evaluate_predictions
from src.features import build_feature_bundle, build_snapshot_feature_bundle
from src.labels import FINAL_LABELS, encode_target
from src.models import build_candidates, compare_models, predict_proba_aligned
from src.preprocessing import preprocess_raw_file
from src.reporting import calibration_parity_report, subgroup_metrics
from src.simulation import reranking_proxy_simulation, suppression_policy_simulation
from src.split import rolling_backtest_splits, temporal_train_validation_test_split
from src.utils import save_csv, save_json


### Function: `run_training_pipeline`

This block is extracted directly from `src/train.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def run_training_pipeline(config: ProjectConfig | None = None) -> Dict[str, object]:
    """Run the final research-grade pipeline.

    This is the single authoritative training path for the dissertation project.
    It avoids duplicate notebook-specific logic and keeps all decisions traceable
    through saved artifacts.
    """

    config = config or ProjectConfig()
    config.ensure_directories()

    preprocessing_result = preprocess_raw_file(config)
    processed = preprocessing_result.frame.copy()

    supervised = processed.loc[processed["outcome_label"].isin(FINAL_LABELS)].copy()
    split_raw = temporal_train_validation_test_split(
        supervised,
        train_fraction=config.train_fraction,
        validation_fraction=config.validation_fraction,
        test_fraction=config.test_fraction,
    )
    save_json(config.reports_dir / "split_metadata.json", split_raw.metadata)

    train_bundle = build_feature_bundle(split_raw.train, include_labels=True)
    validation_bundle = build_snapshot_feature_bundle(split_raw.validation, split_raw.train)
    history_for_test = pd.concat([split_raw.train, split_raw.validation], ignore_index=True)
    test_bundle = build_snapshot_feature_bundle(split_raw.test, history_for_test)

    split = type("FeatureSplit", (), {})()
    split.train = train_bundle.frame.copy()
    split.validation = validation_bundle.frame.copy()
    split.test = test_bundle.frame.copy()
    split.metadata = split_raw.metadata

    feature_bundle = train_bundle
    split.train["target"], target_mapping = encode_target(split.train["outcome_label"])
    split.validation["target"] = split.validation["outcome_label"].map(target_mapping)
    split.test["target"] = split.test["outcome_label"].map(target_mapping)

    full_feature_columns = feature_bundle.feature_groups["full_valid_model"]
    train_X = split.train[full_feature_columns]
    train_y = split.train["target"].to_numpy()
    validation_X = split.validation[full_feature_columns]
    validation_y = split.validation["target"].to_numpy()
    test_X = split.test[full_feature_columns]
    test_y = split.test["target"].to_numpy()

    model_results = compare_models(
        X_train=train_X,
        y_train=train_y,
        X_validation=validation_X,
        y_validation=validation_y,
        labels=FINAL_LABELS,
        categorical_features=[f for f in feature_bundle.categorical_features if f in full_feature_columns],
        numeric_features=[f for f in feature_bundle.numeric_features if f in full_feature_columns],
        alpha=config.calibration_alpha,
        random_state=config.random_state,
    )
    model_rows = _model_result_rows(model_results)
    save_csv(config.reports_dir / "model_comparison.csv", comparison_table(model_rows))
    save_json(config.reports_dir / "model_comparison.json", {"rows": model_rows})

    best_model_result = model_results[0]
    best_estimator = best_model_result.estimator

    calibration_results = compare_calibrators(best_model_result.validation_probabilities, validation_y, FINAL_LABELS)
    calibration_rows = _calibration_result_rows(calibration_results)
    save_csv(config.reports_dir / "calibration_comparison.csv", comparison_table(calibration_rows))
    save_json(config.reports_dir / "calibration_comparison.json", {"rows": calibration_rows})

    best_calibrator_result = calibration_results[0]
    raw_test_probabilities = predict_proba_aligned(best_estimator, test_X, list(range(len(FINAL_LABELS))))
    calibrated_test_probabilities = align_calibrated_probabilities(
        best_calibrator_result.calibrator,
        raw_test_probabilities,
        list(range(len(FINAL_LABELS))),
    )
    final_metrics = evaluate_predictions(test_y, calibrated_test_probabilities, FINAL_LABELS)
    save_json(config.reports_dir / "final_test_metrics.json", final_metrics)

    subgroup_reports = {
        "provider_key": subgroup_metrics(split.test, test_y, calibrated_test_probabilities, "provider_key", FINAL_LABELS),
        "route": subgroup_metrics(split.test, test_y, calibrated_test_probabilities, "route", FINAL_LABELS, top_n=20),
        "airline_code": subgroup_metrics(split.test, test_y, calibrated_test_probabilities, "airline_code", FINAL_LABELS),
        "days_to_departure_bucket": subgroup_metrics(split.test, test_y, calibrated_test_probabilities, "days_to_departure_bucket", FINAL_LABELS),
    }
    save_json(config.reports_dir / "subgroup_evaluation.json", subgroup_reports)
    save_json(
        config.reports_dir / "calibration_parity.json",
        calibration_parity_report(
            split.test,
            test_y,
            calibrated_test_probabilities,
            ["provider_key", "route", "airline_code"],
            FINAL_LABELS,
        ),
    )

    ablation_rows = run_ablation_study(
        train_frame=split.train,
        validation_frame=split.validation,
        feature_bundle=feature_bundle,
        model_name=best_model_result.name,
        random_state=config.random_state,
    )
    save_csv(config.reports_dir / "ablation_results.csv", pd.DataFrame(ablation_rows))
    save_json(config.reports_dir / "ablation_results.json", {"rows": ablation_rows})

    backtest_rows = run_rolling_backtest(
        features=split.train.assign(split_part="train").pipe(
            lambda train_frame: pd.concat(
                [
                    train_frame,
                    split.validation.assign(split_part="validation"),
                    split.test.assign(split_part="test"),
                ],
                ignore_index=True,
            )
        ),
        feature_bundle=feature_bundle,
        model_name=best_model_result.name,
        random_state=config.random_state,
        n_windows=config.rolling_backtest_windows,
    )
    save_csv(config.reports_dir / "rolling_backtest.csv", pd.DataFrame(backtest_rows))
    save_json(config.reports_dir / "rolling_backtest.json", {"rows": backtest_rows})

    policy_results = {
        "suppression_policy": suppression_policy_simulation(split.test, calibrated_test_probabilities, FINAL_LABELS, thresholds=[0.2, 0.4, 0.6, 0.8]),
        "reranking_proxy": reranking_proxy_simulation(split.test, calibrated_test_probabilities, FINAL_LABELS),
    }
    save_json(config.reports_dir / "policy_simulation_results.json", policy_results)

    save_csv(config.reports_dir / "feature_availability_audit.csv", pd.DataFrame(feature_bundle.feature_availability))
    save_json(config.reports_dir / "feature_availability_audit.json", {"rows": feature_bundle.feature_availability})

    ambiguous_experiment = run_ambiguous_handling_experiment(processed, config)
    save_json(config.reports_dir / "ambiguous_handling_experiment.json", ambiguous_experiment)

    feature_inventory = {
        "categorical_features": feature_bundle.categorical_features,
        "numeric_features": feature_bundle.numeric_features,
        "feature_groups": feature_bundle.feature_groups,
    }
    save_json(config.reports_dir / "feature_inventory.json", feature_inventory)

    model_bundle = {
        "model_name": best_model_result.name,
        "estimator": best_estimator,
        "calibrator_name": best_calibrator_result.name,
        "calibrator": best_calibrator_result.calibrator,
        "feature_columns": full_feature_columns,
        "categorical_features": [f for f in feature_bundle.categorical_features if f in full_feature_columns],
        "numeric_features": [f for f in feature_bundle.numeric_features if f in full_feature_columns],
        "target_mapping": target_mapping,
        "labels": FINAL_LABELS,
        "inference_reference": feature_bundle.inference_reference,
        "prediction_time_assumption": config.prediction_time_assumption,
        "config": config.to_dict(),
    }
    joblib.dump(model_bundle, config.models_dir / "final_model_bundle.joblib")

    summary = {
        "best_model": best_model_result.name,
        "best_model_selection_score": best_model_result.selection_score,
        "best_calibrator": best_calibrator_result.name,
        "final_test_macro_f1": final_metrics["macro_f1"],
        "final_test_log_loss": final_metrics["log_loss"],
        "final_test_balanced_accuracy": final_metrics["balanced_accuracy"],
        "final_test_weighted_f1": final_metrics["weighted_f1"],
        "processed_rows": int(len(processed)),
        "supervised_rows": int(len(supervised)),
        "artifacts_dir": str(config.artifacts_dir),
    }
    save_json(config.reports_dir / "summary.json", summary)
    return summary


### Function: `run_ablation_study`

This block is extracted directly from `src/train.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def run_ablation_study(
    train_frame: pd.DataFrame,
    validation_frame: pd.DataFrame,
    feature_bundle,
    model_name: str,
    random_state: int,
) -> List[Dict[str, object]]:
    groups = ["topology_basic", "temporal", "price", "reliability_history", "full_valid_model"]
    rows = []
    for group_name in groups:
        if group_name == "full_valid_model":
            feature_columns = feature_bundle.feature_groups[group_name]
        elif group_name == "temporal":
            feature_columns = feature_bundle.feature_groups["topology_basic"] + feature_bundle.feature_groups["temporal"]
        elif group_name == "price":
            feature_columns = feature_bundle.feature_groups["topology_basic"] + feature_bundle.feature_groups["temporal"] + feature_bundle.feature_groups["price"]
        elif group_name == "reliability_history":
            feature_columns = feature_bundle.feature_groups["topology_basic"] + feature_bundle.feature_groups["temporal"] + feature_bundle.feature_groups["reliability_history"]
        else:
            feature_columns = feature_bundle.feature_groups[group_name]

        estimator = _fit_single_model(model_name, train_frame, validation_frame, feature_columns, feature_bundle, random_state)
        probabilities = predict_proba_aligned(estimator, validation_frame[feature_columns], list(range(len(FINAL_LABELS))))
        metrics = evaluate_predictions(validation_frame["target"].to_numpy(), probabilities, FINAL_LABELS)
        rows.append(
            {
                "feature_group": group_name,
                "macro_f1": metrics["macro_f1"],
                "weighted_f1": metrics["weighted_f1"],
                "balanced_accuracy": metrics["balanced_accuracy"],
                "log_loss": metrics["log_loss"],
                "multiclass_brier": metrics["multiclass_brier"],
                "ece_macro": metrics["ece_macro"],
            }
        )
    return rows


### Function: `run_rolling_backtest`

This block is extracted directly from `src/train.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def run_rolling_backtest(
    features: pd.DataFrame,
    feature_bundle,
    model_name: str,
    random_state: int,
    n_windows: int,
) -> List[Dict[str, object]]:
    rows = []
    base_frame = features.drop(columns=["split_part"], errors="ignore").copy()
    for window in rolling_backtest_splits(base_frame, n_windows=n_windows):
        train_bundle = build_feature_bundle(window["train"], include_labels=True)
        test_bundle = build_snapshot_feature_bundle(window["test"], window["train"])
        train_bundle.frame["target"] = train_bundle.frame["outcome_label"].map({label: idx for idx, label in enumerate(FINAL_LABELS)})
        test_bundle.frame["target"] = test_bundle.frame["outcome_label"].map({label: idx for idx, label in enumerate(FINAL_LABELS)})
        feature_columns = train_bundle.feature_groups["full_valid_model"]
        estimator = _fit_single_model(model_name, train_bundle.frame, test_bundle.frame, feature_columns, train_bundle, random_state)
        probabilities = predict_proba_aligned(estimator, test_bundle.frame[feature_columns], list(range(len(FINAL_LABELS))))
        metrics = evaluate_predictions(test_bundle.frame["target"].to_numpy(), probabilities, FINAL_LABELS)
        rows.append(
            {
                "window_id": window["window_id"],
                "train_start": window["train_start"],
                "train_end": window["train_end"],
                "test_start": window["test_start"],
                "test_end": window["test_end"],
                "macro_f1": metrics["macro_f1"],
                "log_loss": metrics["log_loss"],
                "balanced_accuracy": metrics["balanced_accuracy"],
            }
        )
    return rows


### Function: `run_ambiguous_handling_experiment`

This block is extracted directly from `src/train.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def run_ambiguous_handling_experiment(processed: pd.DataFrame, config: ProjectConfig) -> Dict[str, object]:
    """Compare the default ambiguous-excluded setup with a simple ambiguous-kept experiment.

    This is intentionally lightweight. The dissertation point is to make the
    treatment of ambiguous rows explicit, not to claim a fully noise-robust
    alternate labeling system.
    """

    ambiguous_rows = processed["outcome_label"].eq("ambiguous").sum()
    result = {
        "ambiguous_row_count": int(ambiguous_rows),
        "default_training_policy": "exclude_ambiguous_rows",
        "experimental_policy": "keep_ambiguous_as_fifth_class_for_diagnostic_comparison",
    }

    if ambiguous_rows == 0:
        result["status"] = "skipped_no_ambiguous_rows"
        return result

    frame = processed.copy()
    split_raw = temporal_train_validation_test_split(frame, config.train_fraction, config.validation_fraction, config.test_fraction)
    train_bundle = build_feature_bundle(split_raw.train, include_labels=True)
    validation_bundle = build_snapshot_feature_bundle(split_raw.validation, split_raw.train)
    mapping = {label: idx for idx, label in enumerate(FINAL_LABELS + ["ambiguous"])}
    train_bundle.frame["target_5class"] = train_bundle.frame["outcome_label"].map(mapping)
    validation_bundle.frame["target_5class"] = validation_bundle.frame["outcome_label"].map(mapping)
    feature_columns = train_bundle.feature_groups["full_valid_model"]
    estimator = build_candidates(
        categorical_features=[f for f in train_bundle.categorical_features if f in feature_columns],
        numeric_features=[f for f in train_bundle.numeric_features if f in feature_columns],
        random_state=config.random_state,
    )["catboost"]
    estimator.fit(
        train_bundle.frame[feature_columns],
        train_bundle.frame["target_5class"].to_numpy(),
        cat_features=[f for f in train_bundle.categorical_features if f in feature_columns],
        eval_set=(validation_bundle.frame[feature_columns], validation_bundle.frame["target_5class"].to_numpy()),
    )
    probabilities = predict_proba_aligned(estimator, validation_bundle.frame[feature_columns], list(range(5)))
    metrics = evaluate_predictions(validation_bundle.frame["target_5class"].to_numpy(), probabilities, FINAL_LABELS + ["ambiguous"])
    result["status"] = "completed"
    result["validation_macro_f1_5class"] = metrics["macro_f1"]
    result["validation_log_loss_5class"] = metrics["log_loss"]
    return result


### Function: `_fit_single_model`

This block is extracted directly from `src/train.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def _fit_single_model(model_name: str, train_frame: pd.DataFrame, validation_frame: pd.DataFrame, feature_columns: List[str], feature_bundle, random_state: int):
    candidates = build_candidates(
        categorical_features=[f for f in feature_bundle.categorical_features if f in feature_columns],
        numeric_features=[f for f in feature_bundle.numeric_features if f in feature_columns],
        random_state=random_state,
    )
    estimator = candidates[model_name]
    X_train = train_frame[feature_columns]
    y_train = train_frame["target"].to_numpy()
    X_validation = validation_frame[feature_columns]
    y_validation = validation_frame["target"].to_numpy()
    if model_name == "catboost":
        estimator.fit(
            X_train,
            y_train,
            cat_features=[f for f in feature_bundle.categorical_features if f in feature_columns],
            eval_set=(X_validation, y_validation),
        )
    else:
        estimator.fit(X_train, y_train)
    return estimator


### Function: `_model_result_rows`

This block is extracted directly from `src/train.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def _model_result_rows(results) -> List[Dict[str, object]]:
    rows = []
    for result in results:
        rows.append(
            {
                "name": result.name,
                "selection_score": result.selection_score,
                "accuracy": result.validation_metrics["accuracy"],
                "balanced_accuracy": result.validation_metrics["balanced_accuracy"],
                "macro_f1": result.validation_metrics["macro_f1"],
                "weighted_f1": result.validation_metrics["weighted_f1"],
                "log_loss": result.validation_metrics["log_loss"],
                "multiclass_brier": result.validation_metrics["multiclass_brier"],
                "ece_macro": result.validation_metrics["ece_macro"],
                "tuning_metadata": result.tuning_metadata,
            }
        )
    return rows


### Function: `_calibration_result_rows`

This block is extracted directly from `src/train.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def _calibration_result_rows(results) -> List[Dict[str, object]]:
    rows = []
    for result in results:
        rows.append(
            {
                "name": result.name,
                "selection_score": result.selection_score,
                "accuracy": result.validation_metrics["accuracy"],
                "balanced_accuracy": result.validation_metrics["balanced_accuracy"],
                "macro_f1": result.validation_metrics["macro_f1"],
                "weighted_f1": result.validation_metrics["weighted_f1"],
                "log_loss": result.validation_metrics["log_loss"],
                "multiclass_brier": result.validation_metrics["multiclass_brier"],
                "ece_macro": result.validation_metrics["ece_macro"],
            }
        )
    return rows


### Conditional Block: `__name__ == '__main__'`

This block is extracted directly from `src/train.py` so the implementation can be reviewed in smaller steps.

In [ ]:
if __name__ == "__main__":
    summary = run_training_pipeline()
    print(json.dumps(summary, indent=2))


## Inference Pipeline

**Source file:** `src/inference.py`

This module applies the saved model bundle to new CSV data using the final preprocessing and feature contract.

### Imports, constants, and top-level configuration

This block contains the imports, module-level constants, and shared state used by `src/inference.py`.

In [ ]:
import argparse
import json
from pathlib import Path

import joblib
import pandas as pd

from src.calibration import align_calibrated_probabilities
from src.config import ProjectConfig
from src.features import build_snapshot_feature_bundle
from src.models import predict_proba_aligned
from src.preprocessing import preprocess_raw_frame


### Function: `run_inference`

This block is extracted directly from `src/inference.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def run_inference(input_csv: Path, output_csv: Path, history_csv: Path, bundle_path: Path | None = None) -> pd.DataFrame:
    """Run batch/offline inference with an explicit historical reference dataset.

    This entry point is intentionally honest in scope. It is not a live online
    service with a feature store. Instead, it requires a historical labeled
    dataset so that the same history-based feature logic used in validation and
    test can be reproduced during offline scoring.
    """

    config = ProjectConfig()
    bundle_path = bundle_path or (config.models_dir / "final_model_bundle.joblib")
    bundle = joblib.load(bundle_path)

    raw_input = pd.read_csv(input_csv)
    raw_history = pd.read_csv(history_csv)

    if "Status" not in raw_input.columns and "status" not in raw_input.columns:
        raw_input["Status"] = "unknown"
    if "Status" not in raw_history.columns and "status" not in raw_history.columns:
        raise ValueError("History CSV must include raw outcome labels so prior-only history features can be built safely.")

    input_preprocessed = preprocess_raw_frame(raw_input, config).frame
    history_preprocessed = preprocess_raw_frame(raw_history, config).frame

    scored_bundle = build_snapshot_feature_bundle(input_preprocessed, history_preprocessed)
    X = scored_bundle.frame[bundle["feature_columns"]]

    probabilities = predict_proba_aligned(bundle["estimator"], X, list(range(len(bundle["labels"]))))
    calibrated = align_calibrated_probabilities(bundle["calibrator"], probabilities, list(range(len(bundle["labels"]))))

    prediction_frame = pd.DataFrame(calibrated, columns=[f"p_{label}" for label in bundle["labels"]])
    prediction_frame["predicted_label"] = prediction_frame.idxmax(axis=1).str.replace("p_", "", regex=False)

    output = pd.concat(
        [
            scored_bundle.frame[
                [
                    "prediction_time",
                    "departure_date",
                    "origin_airport",
                    "destination_airport",
                    "airline_code",
                    "provider_key",
                    "meta_engine",
                    "search_group_proxy",
                ]
            ].reset_index(drop=True),
            prediction_frame,
        ],
        axis=1,
    )
    output.to_csv(output_csv, index=False)
    return output


### Function: `main`

This block is extracted directly from `src/inference.py` so the implementation can be reviewed in smaller steps.

In [ ]:
def main() -> None:
    parser = argparse.ArgumentParser(description="Run the final offline batch inference pipeline.")
    parser.add_argument("--input", required=True, help="Path to the input CSV.")
    parser.add_argument("--history", required=True, help="Path to the historical reference CSV.")
    parser.add_argument("--output", required=True, help="Path to the output predictions CSV.")
    parser.add_argument("--bundle", required=False, help="Optional path to the model bundle.")
    args = parser.parse_args()

    output = run_inference(
        input_csv=Path(args.input),
        output_csv=Path(args.output),
        history_csv=Path(args.history),
        bundle_path=Path(args.bundle) if args.bundle else None,
    )
    print(json.dumps({"rows_scored": len(output), "output_path": args.output}, indent=2))


### Conditional Block: `__name__ == '__main__'`

This block is extracted directly from `src/inference.py` so the implementation can be reviewed in smaller steps.

In [ ]:
if __name__ == "__main__":
    main()
